# Anomaly detection for visual inspection (MVTec AD)

[MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad) is a benchmark of 15 categories (5 textures, 10 objects) photographed for industrial inspection. The key property: **the training set contains only good images**. A model has to learn what "normal" looks like and flag anything that deviates, without ever seeing a defect.

This notebook compares a classical OpenCV golden-template baseline with deep methods (via anomalib).

*The write-up with the main results is the last section of this notebook.*

## Loading and exploring the dataset

Imports and settings. Everything that can be tuned is collected in this one cell, so the notebook is easy to reproduce.

In [ ]:
import os
from pathlib import Path
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# --- data ---
# folder with one subfolder per category (how to download: see the Data section); set MVTEC_AD_ROOT or edit this line
PATH_ROOT = Path(os.environ.get("MVTEC_AD_ROOT", "datasets/mvtec_ad_dataset"))
RESOLUTION = 512  # images are stored at this size (original is 1024x1024)
CATEGORIES = ["transistor", "pill", "metal_nut", "leather", "carpet", "grid"]  # used in the step-by-step comparison

# --- classical pipeline ---
BLUR = 7           # Gaussian blur on the z-map (kernel size)
PCTL = 99.5        # image score = this percentile of the blurred z-map
STD_FLOOR = 1e-3   # avoids divide-by-zero where the good images never vary
Z_THRESH = 4       # pixel is anomalous above this many std (masks / visualisation only, not the AUROC)
KERNEL = 3         # morphology kernel size for cleaning the mask

In [ ]:
# PatchCore further down needs anomalib (versions of all packages: requirements.txt). Installed only if it is missing.
import importlib.util

if importlib.util.find_spec("anomalib") is None:
    %pip install anomalib==2.7.0

First, load the images from the dataset folder. The folder structure is `category/split/label/image.png`, and the loop below turns that into four parallel arrays:

- `X`: the images, converted from OpenCV's BGR to RGB, resized to `RESOLUTION` and scaled to 0–1
- `labels`: `good` or the defect type (e.g. `bent_lead`)
- `categories`: the object or texture (carpet, grid...)
- `splits`: `train` (good images only) or `test` (good and defective)

The `ground_truth` folders hold the defect masks, not images to score, so they are skipped here. They are only needed later for pixel-level evaluation.

The images are 1024×1024 originally. Keeping them at 512×512 leaves room to test lower resolutions later without upsampling.

In [ ]:
X, labels, categories, splits = [], [], [], []

for img_path in PATH_ROOT.rglob("*.png"):
    label = img_path.parent.name
    split = img_path.parent.parent.name
    category = img_path.parent.parent.parent.name
    if split == "ground_truth":
        continue  # masks are not images to score; load them separately for pixel-level evaluation
    if category not in CATEGORIES:
        continue  # only the categories used in this notebook
    img = cv2.imread(str(img_path))  # BGR by default
    if img is None:
        continue  # skip unreadable files

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (RESOLUTION, RESOLUTION))
    img = img.astype("float32") / 255.0  # normalize our values between 0 and 1

    X.append(img)
    labels.append(label)
    categories.append(category)
    splits.append(split)

if not X:
    raise FileNotFoundError(f"no images found under {PATH_ROOT.resolve()}; download MVTec AD (see the Data section) and set MVTEC_AD_ROOT or PATH_ROOT")

X = np.array(X)
labels = np.array(labels)
categories = np.array(categories)
splits = np.array(splits)

Check which categories were actually loaded. `X` holds every image of those categories in memory, so loading all 15 at this resolution would need many GB:

In [ ]:
np.unique(categories)


Plot a random sample of images to get a feeling for what is in the dataset. Each title shows `label | category | split`:

- the **textures** (carpet, leather, grid...) have no fixed pose, only a repeating surface pattern
- the **objects** (transistor, pill, metal_nut...) are single parts photographed on a plain background, some always in the same position, some rotated

Run the cell a few times (the sampling is random) and look at what a defect actually looks like compared to a good image. Some are obvious (missing parts), some are barely visible (small scratches).

In [ ]:
import matplotlib.pyplot as plt

n = 9  # number of images to show
rng = np.random.default_rng()  # add seed for reproducibvle random run

idx = rng.choice(len(X), size=n, replace=False)

plt.figure(figsize=(12, 12))

for i, image_idx in enumerate(idx):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X[image_idx])
    plt.axis("off")
    plt.title(
        f"{labels[image_idx]} | {categories[image_idx]} | {splits[image_idx]}\nidx {image_idx}",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

## Classical baseline: golden template (OpenCV)

A deliberately simple baseline with no learning: compare each test image with an "average good image" and score how far it deviates. This is how many traditional inspection systems work, and it sets the bar that the deep methods later have to beat.

How could we do a simple anomaly detection in OpenCV per category?

The classical idea is a **golden template**: build a reference of what "good" looks like from the good images, and flag whatever differs from it. For example, use the average of all good images. This only works if the good images agree pixel by pixel, which is not true for every category (images of grids have different alignments and orientations, and textures only agree in their statistics).

A practical recipe for MVTec-like data:
1. Align images (same object pose).
2. Build a "normal" reference (mean image or per-pixel stats).
3. Compute the difference / z-score.
4. Threshold + morphology.
5. Find contours and filter by size/shape.
6. Visualize an overlay of the anomaly mask on the original image.

The next cells implement steps 2–4 and evaluate them. Step 1 (alignment) is added later in the step-by-step section, and steps 5–6 are left out of this baseline.

Helper functions for the whole classical pipeline, defined once and reused by every experiment below:

- `select`: indices of the images of a category / split / good-or-defective
- `build_model`: per-pixel mean and std of the good training images, optionally after ECC alignment
- `zmap` / `score`: z-score map of one image, and the single number derived from it
- `metrics`: image AUROC and AUROC per defect type. It only needs one score per image, so the deep methods later can be evaluated with exactly the same function
- `evaluate`: all of the above for one category, plus time per image

In [ ]:
def select(ctg, split, good=None):
    """Indices into X for one category and split. good=True/False keeps only good/defective images."""
    m = (categories == ctg) & (splits == split)
    if good is not None:
        m &= (labels == "good") == good
    return np.where(m)[0]


def prep(img, res):
    """RGB float image (RESOLUTION x RESOLUTION) -> grayscale at `res`."""
    assert res <= RESOLUTION, "cannot upsample: X is stored at RESOLUTION"
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    if g.shape[0] != res:
        g = cv2.resize(g, (res, res), interpolation=cv2.INTER_AREA)
    return g


def align_ecc(gray, template, n_iter=100, eps=1e-5):
    """Rotation + translation alignment of `gray` onto `template`. Returns (aligned, converged)."""
    warp = np.eye(2, 3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, n_iter, eps)
    try:
        _, warp = cv2.findTransformECC(template, gray, warp, cv2.MOTION_EUCLIDEAN, criteria, None, 5)
    except cv2.error:
        return gray, False
    h, w = template.shape
    aligned = cv2.warpAffine(
        gray, warp, (w, h),
        flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
        borderMode=cv2.BORDER_REFLECT,
    )
    return aligned, True


def build_model(ctg, res=256, align=False, use_std=True, aligner=align_ecc, roi=None):
    """Per-pixel mean/std of the good training images (the golden template)."""
    train = [prep(im, res) for im in X[select(ctg, "train", good=True)]]
    tmpl = train[0]  # a sharp real image is a more stable alignment target than the blurry mean
    if align:
        train = [aligner(g, tmpl)[0] for g in train]
    train = np.stack(train)
    mean, std = train.mean(0), train.std(0)
    if use_std:
        std = std + STD_FLOOR
    else:
        std = np.full_like(mean, std.mean() + STD_FLOOR)  # constant -> plain absdiff, just rescaled
    mask = None
    if roi:  # circular region of interest (fraction of the image radius); ignores corners the rotation leaves empty
        yy, xx = np.mgrid[:res, :res]
        mask = (yy - res / 2) ** 2 + (xx - res / 2) ** 2 <= (roi * res / 2) ** 2
    return dict(res=res, align=align, aligner=aligner, mean=mean, std=std, tmpl=tmpl, mask=mask)


def zmap(model, img):
    """Blurred per-pixel z-score of one image against the model."""
    g = prep(img, model["res"])
    if model["align"]:
        g, _ = model["aligner"](g, model["tmpl"])
    z = np.abs(g - model["mean"]) / model["std"]
    return cv2.GaussianBlur(z, (BLUR, BLUR), 0)


def score(model, img):
    """One number per image: high = more anomalous."""
    z = zmap(model, img)
    if model["mask"] is not None:
        z = z[model["mask"]]  # only score inside the region of interest
    return np.percentile(z, PCTL)


def metrics(y, s, lab):
    """Image AUROC and AUROC per defect type. Works for any method that gives one score per image.
    y: 1 = defective, s: scores, lab: label per image ("good" or the defect type)."""
    per_defect = {}
    for d in np.unique(lab):
        if d == "good":
            continue
        m = (lab == d) | (lab == "good")
        per_defect[str(d)] = roc_auc_score(lab[m] != "good", s[m])
    return roc_auc_score(y, s), per_defect


def evaluate(ctg, model=None, **opts):
    """Score all test images of a category with the classical method; returns metrics and timing."""
    model = model or build_model(ctg, **opts)
    idx = select(ctg, "test")
    y = (labels[idx] != "good").astype(int)

    t0 = time.perf_counter()
    s = np.array([score(model, X[i]) for i in idx])
    s_per_img = (time.perf_counter() - t0) / len(idx)

    auroc, per_defect = metrics(y, s, labels[idx])
    return dict(auroc=auroc, s_per_img=s_per_img, per_defect=per_defect, idx=idx, y=y, scores=s)

### First attempt on one category

Start with an aligned object, like transistor. From the **good training images only**, compute two things for every pixel:

- the **mean image** `mean_good`: what a normal part looks like (the "golden template")
- the **standard deviation** `std_good`: how much each pixel normally varies

Anything far from the template, relative to its normal variation, is a candidate anomaly.

In [ ]:
ctg = "transistor"
model = build_model(ctg, res=256)  # mean and std of the good training images (grayscale)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(model["mean"], cmap="gray"); ax[0].set_title("mean of the good images")
ax[1].imshow(model["std"], cmap="hot");   ax[1].set_title("per-pixel std")
for a in ax:
    a.axis("off")
plt.show()

Take one random good *training* image and one random defective *test* image, and subtract the average image from each. The raw difference should be dark for the good image and bright where the defective one deviates from normal. (`show_comparison` is a small helper that plots original / reference / difference side by side for both.)

In [ ]:
def show_comparison(rows, third="difference"):
    """rows: list of (original, reference, map) triples, one row of plots each."""
    fig, ax = plt.subplots(len(rows), 3, figsize=(12, 4 * len(rows)), squeeze=False)
    for r, row in enumerate(rows):
        for a, im, title, cmap in zip(ax[r], row, ("original", "reference", third), ("gray", "gray", "hot")):
            a.imshow(im, cmap=cmap)
            a.set_title(title)
            a.axis("off")
    plt.tight_layout()
    plt.show()


good_i = np.random.choice(select(ctg, "train", good=True))
bad_i = np.random.choice(select(ctg, "test", good=False))

g_good, g_bad = prep(X[good_i], model["res"]), prep(X[bad_i], model["res"])
show_comparison([
    (g_good, model["mean"], cv2.absdiff(g_good, model["mean"])),
    (g_bad, model["mean"], cv2.absdiff(g_bad, model["mean"])),
])

A raw difference image is hard to threshold. Turn it into an **anomaly map** in four steps (the first two are `zmap`):

1. **z-score**: `|pixel - mean| / std`, the distance from the normal value in units of that pixel's own normal variation. Dividing by the std makes naturally variable pixels (edges, texture) count less. Build the model with `use_std=False` to fall back to a plain difference.
2. **Blur** (`BLUR`): averages neighbouring pixels so isolated noisy pixels do not look like defects.
3. **Threshold** (`Z_THRESH`): a pixel is anomalous if it is more than this many standard deviations away from normal. The same fixed value is used for every image.
4. **Morphology** (`KERNEL`): opening removes small specks, closing fills small holes, leaving connected regions.

The good image should give a nearly empty mask and the defective one a blob where the defect is.

In [ ]:
def anomaly_map(model, img):
    z = zmap(model, img)
    _, mask = cv2.threshold(z, Z_THRESH, 1, cv2.THRESH_BINARY)

    # clean small noise
    kernel = np.ones((KERNEL, KERNEL), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return z, mask


_, mask_good = anomaly_map(model, X[good_i])
_, mask_bad = anomaly_map(model, X[bad_i])
# print("mask pixels  good:", np.count_nonzero(mask_good), " defective:", np.count_nonzero(mask_bad))

show_comparison([
    (g_good, model["mean"], mask_good),
    (g_bad, model["mean"], mask_bad),
], third="mask")

The masks are good for looking at, but a decision on a production line needs **one number per image** ("is this part good or not?"). Here the image score is the 99.5th percentile of the blurred z-map: a robust version of "how bad is the worst region", which ignores the few most extreme pixels.

To compare scores without choosing a threshold we use the **AUROC (Area Under the Receiver Operating Characteristic curve)** : the probability that a random defective image gets a higher score than a random good one (1.0 = perfect ranking, 0.5 = coin flip). It is also broken down per defect type, to see which defects the method misses.

In [ ]:
r = evaluate(ctg, model=model)
print(f"{ctg}: image AUROC = {r['auroc']:.3f}  ({r['y'].sum()} defective, {(1 - r['y']).sum()} good)")
for d, a in r["per_defect"].items():
    print(f"  {d}: {a:.3f}")

What the AUROC looks like. Each image has one score, so we can plot the scores of the good and the defective images (left). If the two distributions barely overlap, the score separates them well. The **ROC curve** (right) comes from sweeping a threshold across the left plot: for each threshold it shows the fraction of defects caught (true positive rate) against the fraction of good parts wrongly rejected (false positive rate). The **AUROC** is the area under that curve.

In symbols, with `s₊` the score of a random defective image and `s₋` that of a random good one: `AUROC = P(s₊ > s₋)`.

The dashed line and red dot mark one operating point: the threshold that rejects at most `TARGET_FPR` of the good parts. Moving it along the curve is the false-reject versus escape trade-off on a real line.

In [ ]:
from sklearn.metrics import roc_curve

TARGET_FPR = 0.01  # operating point: fraction of good parts we are willing to reject

y, s = r["y"], r["scores"]
fpr, tpr, thr = roc_curve(y, s)

# threshold with the highest TPR that keeps the false-reject rate <= TARGET_FPR
k = np.where(fpr <= TARGET_FPR)[0][-1]
print(f"threshold {thr[k]:.2f}: rejects {fpr[k]:.0%} of good parts, catches {tpr[k]:.0%} of defects")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# left: score distributions of good vs defective images, with the chosen threshold
bins = np.linspace(s.min(), s.max(), 30)
ax[0].hist(s[y == 0], bins, alpha=0.6, label="good")
ax[0].hist(s[y == 1], bins, alpha=0.6, label="defective")
ax[0].axvline(thr[k], color="k", ls="--", label="threshold")
ax[0].set_xlabel("anomaly score"); ax[0].set_ylabel("images"); ax[0].legend()
ax[0].set_title("The more they overlap, the lower the AUROC")

# right: ROC curve with the area shaded and the operating point marked
ax[1].plot(fpr, tpr, label=f"AUROC = {r['auroc']:.3f}")
ax[1].fill_between(fpr, tpr, alpha=0.2)
ax[1].plot([0, 1], [0, 1], "k--", label="chance (0.5)")
ax[1].plot(fpr[k], tpr[k], "ro", label="chosen threshold")
ax[1].set_xlabel("false positive rate (good parts rejected)")
ax[1].set_ylabel("true positive rate (defects caught)")
ax[1].legend(loc="lower right"); ax[1].set_title("ROC curve")

plt.tight_layout()
plt.show()

Does dividing by the standard deviation help? Compare two variants on the same test images:

- **plain diff**: every pixel is weighted equally (the std is replaced by a constant, which only rescales the difference)
- **per-pixel z**: every pixel is weighted by how much it normally varies

The printed AUROC pair gives the numbers. The heatmaps show one random defective image under both variants. With plain diff, bright bands along edges and variable regions are false anomalies. With the per-pixel z-score they should be quieter and the defect stands out more. Rerun the cell a few times to look at different defects.

In [ ]:
variants = {"plain diff": dict(use_std=False), "per-pixel z": dict(use_std=True)}
models = {n: build_model(ctg, res=256, **o) for n, o in variants.items()}
results = {n: evaluate(ctg, model=m) for n, m in models.items()}
print(f"{ctg}  " + "   ".join(f"{n}: {r['auroc']:.3f}" for n, r in results.items()))

# same defective test image under both variants (each map on its own color scale,
# because dividing by the std changes the units)
bad_i = np.random.choice(select(ctg, "test", good=False))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(prep(X[bad_i], 256), cmap="gray"); ax[0].set_title(labels[bad_i])
for a, (n, m) in zip(ax[1:], models.items()):
    a.imshow(zmap(m, X[bad_i]), cmap="hot"); a.set_title(n)
for a in ax:
    a.axis("off")
plt.show()

### Building it up step by step

Same pipeline for every step, one change at a time. Categories: transistor (aligned, rigid), pill (aligned shape, random speckles), leather (pure texture).

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead><tr>
<th style="text-align: left;">Step</th>
<th style="text-align: left;">Change</th>
<th style="text-align: left;">Question</th>
</tr></thead>
<tbody>
<tr><td style="text-align: left;">1</td><td style="text-align: left;">plain absdiff to the mean image</td><td style="text-align: left;">what does a golden template give for free?</td></tr>
<tr><td style="text-align: left;">2</td><td style="text-align: left;">per-pixel z-score (divide by std)</td><td style="text-align: left;">does modeling normal variability help?</td></tr>
<tr><td style="text-align: left;">3</td><td style="text-align: left;">ECC alignment (train and test)</td><td style="text-align: left;">how much of the error was misalignment?</td></tr>
<tr><td style="text-align: left;">4</td><td style="text-align: left;">resolution 256 → 512</td><td style="text-align: left;">were small defects lost to downsampling?</td></tr>
</tbody>
</table>

Fixed throughout: grayscale, Gaussian blur 7, image score = 99.5th percentile of the z-map. Nothing is tuned on the test set.

In [ ]:
# Each step changes exactly one thing relative to the previous one.
# To add a step, add a line here and rerun this cell: finished (step, category) pairs are skipped.
STEPS = {
    "1 plain diff":       dict(use_std=False, align=False, res=256),
    "2 + per-pixel z":    dict(use_std=True,  align=False, res=256),
    "3 + ECC align":      dict(use_std=True,  align=True,  res=256),
    "4 + resolution 512": dict(use_std=True,  align=True,  res=512),
}

try:
    RESULTS
except NameError:
    RESULTS = {}

for step, opts in STEPS.items():
    for ctg in CATEGORIES:
        if (step, ctg) in RESULTS:
            continue
        RESULTS[(step, ctg)] = evaluate(ctg, **opts)
        print(f"{step:20s} {ctg:12s} AUROC {RESULTS[(step, ctg)]['auroc']:.3f}  "
              f"{RESULTS[(step, ctg)]['s_per_img']:.3f} s/img")

In [ ]:
import pandas as pd

rows = [
    (step, ctg, r["auroc"], r["s_per_img"])
    for (step, ctg), r in RESULTS.items()
]
df = pd.DataFrame(rows, columns=["step", "category", "auroc", "s_per_img"])

print("Image AUROC")
print(df.pivot(index="step", columns="category", values="auroc").round(3))
print("\nSeconds per image (scoring only)")
print(df.pivot(index="step", columns="category", values="s_per_img").round(3))

# per-defect-type AUROC for the last step, to see which defects are missed
last = list(STEPS)[-1]
for ctg in CATEGORIES:
    if (last, ctg) in RESULTS:
        print(f"\n{ctg} | {last}:", {k: round(v, 3) for k, v in RESULTS[(last, ctg)]["per_defect"].items()})

df.to_csv("classical_ablation.csv", index=False)

In [ ]:
def show_examples(ctg, step):
    """False alarm / miss / hit for one category and step, using the stored scores."""
    r = RESULTS[(step, ctg)]
    model = build_model(ctg, **STEPS[step])
    idx, y, s = r["idx"], r["y"], r["scores"]
    picks = {
        "false alarm (good, highest)": idx[y == 0][np.argmax(s[y == 0])],
        "miss (defect, lowest)":       idx[y == 1][np.argmin(s[y == 1])],
        "hit (defect, highest)":       idx[y == 1][np.argmax(s[y == 1])],
    }
    maps = {t: zmap(model, X[i]) for t, i in picks.items()}
    vmax = max(m.max() for m in maps.values())  # shared scale so the three are comparable

    fig, ax = plt.subplots(2, 3, figsize=(12, 8))
    for j, (title, i) in enumerate(picks.items()):
        sc = s[np.where(idx == i)[0][0]]
        ax[0, j].imshow(X[i])
        ax[0, j].set_title(f"{title}\n{labels[i]}, score {sc:.1f}", fontsize=9)
        ax[1, j].imshow(maps[title], cmap="hot", vmax=vmax)
    for a in ax.ravel():
        a.axis("off")
    fig.suptitle(f"{ctg} | {step}")
    plt.tight_layout()
    plt.show()

show_examples("transistor", "3 + ECC align")

## Side experiment: rotated parts

ECC alignment starts from the identity transform and only converges for small misalignments. `metal_nut` is photographed at arbitrary rotations, so plain ECC should help little there. Here the same pipeline gets a **multi-start ECC**: ECC is run from many initial rotation angles and the alignment with the best correlation is kept. This is a side question, not part of the main comparison, and it costs one ECC run per start angle.

Rotating the square image leaves empty corners that `BORDER_REFLECT` fills with mirrored content. The last variant scores only inside a circle (`roi=0.9`) that the rotated frame always covers, so those artifacts cannot reach the score.


In [ ]:
def align_multistart(gray, template, step=20):
    """ECC from many initial rotations; keep the alignment with the best correlation."""
    h, w = template.shape
    crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-5)
    best_cc, best_warp = -1.0, None
    for a in range(0, 360, step):
        warp = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0).astype(np.float32)
        try:
            cc, warp = cv2.findTransformECC(template, gray, warp, cv2.MOTION_EUCLIDEAN, crit, None, 5)
        except cv2.error:
            continue
        if cc > best_cc:
            best_cc, best_warp = cc, warp
    if best_warp is None:
        return gray, False
    aligned = cv2.warpAffine(gray, best_warp, (w, h),
                             flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REFLECT)
    return aligned, True


# look at the alignment itself: template, raw image, plain ECC, multi-start ECC
ctg_side = "metal_nut"
tmpl = build_model(ctg_side, res=256)["tmpl"]
picks = np.random.choice(select(ctg_side, "test", good=True), 3, replace=False)

fig, ax = plt.subplots(len(picks), 4, figsize=(12, 3 * len(picks)))
for r_, i in enumerate(picks):
    g = prep(X[i], 256)
    for a, im, t in zip(ax[r_], (tmpl, g, align_ecc(g, tmpl)[0], align_multistart(g, tmpl)[0]),
                        ("template", "raw", "ECC", "multi-start ECC")):
        a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
side_variants = {
    "z, no alignment":    dict(align=False),
    "z + ECC":            dict(align=True),
    "z + multi-start ECC": dict(align=True, aligner=align_multistart),
    "z + multi-start ECC + circular ROI": dict(align=True, aligner=align_multistart, roi=0.9),
}
side = {n: evaluate(ctg_side, res=256, **o) for n, o in side_variants.items()}

for n, r_ in side.items():
    print(f"{n:36s} AUROC {r_['auroc']:.3f}   {r_['s_per_img']:.3f} s/img")
    print("   ", {k: round(v, 3) for k, v in r_["per_defect"].items()})

## Side experiments: other structures

Two quick looks at how other kinds of surfaces are usually handled. They use a different score per category and the same metrics, so the AUROC values are comparable with the template results above. These are for interest, so each one is a single simple version, with parameters fixed in advance and not tuned on the test set.

**Textures without a template.** Leather has no repeating pattern, but carpet and grid are periodic. Two scores that need no good-image model:

- **high-pass**: subtract a blurred copy of the image from itself. Normal texture mostly cancels, and a local defect remains.
- **Fourier notch**: a periodic texture shows up as sharp peaks in the 2D spectrum. Remove those peaks and transform back, and what is left is mostly the part that does not repeat (the defect).

The template score from the step-by-step section (per-pixel z, no alignment) is shown for reference.

In [ ]:
def evaluate_fn(ctg, fn):
    """Evaluate any per-image scoring function fn(rgb image) -> float with the same metrics as the template method."""
    idx = select(ctg, "test")
    y = (labels[idx] != "good").astype(int)
    t0 = time.perf_counter()
    s = np.array([fn(X[i]) for i in idx])
    s_per_img = (time.perf_counter() - t0) / len(idx)
    auroc, per_defect = metrics(y, s, labels[idx])
    return dict(auroc=auroc, s_per_img=s_per_img, per_defect=per_defect)


def highpass_parts(img, sigma=8):
    g = prep(img, 256)
    blur = cv2.GaussianBlur(g, (0, 0), sigma)
    resid = cv2.GaussianBlur(np.abs(g - blur), (BLUR, BLUR), 0)
    return g, blur, resid


def score_highpass(img, sigma=8):
    return np.percentile(highpass_parts(img, sigma)[2], PCTL)


def fourier_parts(img, k=5):
    """Remove spectral peaks (the repeating pattern); what is left is the residual."""
    g = prep(img, 256)
    gw = (g - g.mean()) * np.outer(np.hanning(256), np.hanning(256))  # window: less leakage from the image border
    F = np.fft.fftshift(np.fft.fft2(gw))
    M = np.abs(F).astype(np.float32)
    peaks = M > k * cv2.GaussianBlur(M, (0, 0), 3)  # much stronger than the local spectrum level
    F[peaks] = 0
    res = np.abs(np.fft.ifft2(np.fft.ifftshift(F)))[32:-32, 32:-32]  # crop where the window suppressed the image
    return g, M, peaks, cv2.GaussianBlur(res.astype(np.float32), (BLUR, BLUR), 0)


def score_fourier(img, k=5):
    return np.percentile(fourier_parts(img, k)[3], PCTL)


tex = {}
for ctg_t in ["leather", "carpet", "grid"]:
    tex[ctg_t] = {
        "template z":  evaluate(ctg_t, res=256),
        "high-pass":   evaluate_fn(ctg_t, score_highpass),
        "Fourier notch": evaluate_fn(ctg_t, score_fourier),
    }

print(pd.DataFrame({c: {m: r["auroc"] for m, r in d.items()} for c, d in tex.items()}).round(3))

In [ ]:
def show_texture_steps(ctg, i=None):
    """Every step of the two texture scores on one defective image."""
    i = np.random.choice(select(ctg, "test", good=False)) if i is None else i
    g, blur, hp = highpass_parts(X[i])
    _, M, peaks, fres = fourier_parts(X[i])
    spec = np.log1p(M)
    overlay = np.dstack([spec / spec.max()] * 3)
    overlay[cv2.dilate(peaks.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0] = (1, 0, 0)  # removed peaks in red

    panels = [
        (g, "gray", "image"), (blur, "gray", "blurred copy"), (hp, "hot", "high-pass residual"),
        (spec, "gray", "log spectrum"), (overlay, None, f"peaks removed ({peaks.sum()}, red)"), (fres, "hot", "Fourier residual"),
    ]
    fig, ax = plt.subplots(2, 3, figsize=(12, 8))
    for a, (im, cmap, t) in zip(ax.ravel(), panels):
        a.imshow(im, cmap=cmap); a.set_title(t); a.axis("off")
    fig.suptitle(f"{ctg}: {labels[i]}")
    plt.tight_layout()
    plt.show()


show_texture_steps("grid")

**Pill: does color help?** The pill is a fixed shape with random red speckles that are normal. Everything below is the same per-pixel template (mean and std of the good images, no alignment), changing one thing at a time:

- **gray (L)**: lightness only, as in the main pipeline
- **Lab**: adds the two color channels (the worst of L, a, b is scored). The speckles are red, and many defects are not
- **+ pill mask**: score only inside the pill (Otsu on the dark background), away from its edge
- **+ speckle filter**: median filter before comparing, so marks smaller than about 11 px disappear

In [ ]:
def pill_prep(img, res=256, k=1):
    rgb8 = (cv2.resize(img, (res, res), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
    lab = cv2.cvtColor(rgb8, cv2.COLOR_RGB2LAB)
    _, m = cv2.threshold(lab[..., 0], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = cv2.erode(m, np.ones((9, 9), np.uint8)) > 0  # stay away from the pill edge
    if k > 1:
        lab = cv2.medianBlur(lab, k)  # removes marks smaller than ~k pixels
    return lab.astype(np.float32), m


def build_pill_model(ctg, channels=(0, 1, 2), k=1, lab_floor=2.0):
    ch = list(channels)
    stack = np.stack([pill_prep(im, k=k)[0][..., ch] for im in X[select(ctg, "train", good=True)]])
    return dict(ch=ch, k=k, mean=stack.mean(0), std=stack.std(0) + lab_floor)


def pill_zmap(model, img):
    lab, m = pill_prep(img, k=model["k"])
    z = (np.abs(lab[..., model["ch"]] - model["mean"]) / model["std"]).max(axis=2)  # worst channel
    return cv2.GaussianBlur(z, (BLUR, BLUR), 0), m


def make_pill_score(ctg, use_mask=False, **kw):
    model = build_pill_model(ctg, **kw)

    def fn(img):
        z, m = pill_zmap(model, img)
        return np.percentile(z[m] if use_mask else z, PCTL)
    return fn


pill_variants = {
    "gray (L)":             dict(channels=(0,)),
    "Lab":                  dict(),
    "Lab + pill mask":      dict(use_mask=True),
    "Lab + speckle filter": dict(k=11),
}
pill = {n: evaluate_fn("pill", make_pill_score("pill", **o)) for n, o in pill_variants.items()}

print(pd.DataFrame({n: {"image": r["auroc"], **r["per_defect"]} for n, r in pill.items()}).round(2))

In [ ]:
def show_pill_steps(i=None, defect=None):
    """Every step of the pill comparison on one defective image."""
    if i is None:
        pool = select("pill", "test", good=False)
        if defect:
            pool = pool[labels[pool] == defect]
        i = np.random.choice(pool)
    raw, _ = pill_prep(X[i])
    filt, mask = pill_prep(X[i], k=11)
    z_gray, _ = pill_zmap(build_pill_model("pill", channels=(0,)), X[i])
    z_lab, _ = pill_zmap(build_pill_model("pill"), X[i])
    to_rgb = lambda lab: cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)

    panels = [
        (to_rgb(raw), None, "image"), (raw[..., 0], "gray", "L (lightness)"),
        (raw[..., 1], "RdBu_r", "a (green to red): speckles show here"), (raw[..., 2], "RdBu_r", "b (blue to yellow)"),
        (to_rgb(filt), None, "after median filter (k=11)"), (mask, "gray", "pill mask"),
        (z_gray, "hot", "z-map, gray only"), (z_lab, "hot", "z-map, Lab (worst channel)"),
    ]
    fig, ax = plt.subplots(2, 4, figsize=(16, 8))
    for a, (im, cmap, t) in zip(ax.ravel(), panels):
        a.imshow(im, cmap=cmap); a.set_title(t, fontsize=10); a.axis("off")
    fig.suptitle(f"pill: {labels[i]}")
    plt.tight_layout()
    plt.show()


show_pill_steps(defect="color")

**Gabor filters, as an illustration.** For textures without a repeating pattern, such as leather, a common classical description is a bank of Gabor filters. A Gabor kernel is a stripe pattern at one orientation and spacing, faded out by a Gaussian window. Sliding it over the image gives a map that is large wherever the local texture has stripes matching it. Many kernels together describe a patch by which orientations and scales are present, and a defect changes that description even when the brightness does not. This cell only shows the kernels and their responses on a defective leather image; there is no score here.

In [ ]:
def show_gabor(ctg="leather", i=None, wavelengths=(8, 16), thetas=(0, 45, 90, 135)):
    i = np.random.choice(select(ctg, "test", good=False)) if i is None else i
    g = prep(X[i], 256)

    def kernel(lambd, theta):
        k = cv2.getGaborKernel((41, 41), 0.56 * lambd, np.deg2rad(theta), lambd, 0.5, 0)
        return k / np.abs(k).sum()

    fig, ax = plt.subplots(1 + len(wavelengths), len(thetas) + 1, figsize=(15, 3.2 * (1 + len(wavelengths))))
    ax[0, 0].imshow(g, cmap="gray"); ax[0, 0].set_title(f"{ctg}: {labels[i]}")
    for c, th in enumerate(thetas, start=1):  # row 0: the kernels (shortest wavelength)
        ax[0, c].imshow(kernel(wavelengths[0], th), cmap="gray")
        ax[0, c].set_title(f"kernel: wavelength {wavelengths[0]}px, {th} deg")
    for r, lam in enumerate(wavelengths, start=1):  # rows 1..: responses
        ax[r, 0].axis("off")
        ax[r, 0].text(0.5, 0.5, f"responses\nwavelength {lam}px", ha="center", va="center", fontsize=12)
        for c, th in enumerate(thetas, start=1):
            ax[r, c].imshow(np.abs(cv2.filter2D(g, cv2.CV_32F, kernel(lam, th))), cmap="hot")
            ax[r, c].set_title(f"{th} deg")
    for a in ax.ravel():
        a.axis("off") if a.get_title() == "" else a.set_xticks([]) or a.set_yticks([])
    plt.tight_layout()
    plt.show()


show_gabor()

**Gabor energy + Mahalanobis.** The scoring step that goes with those filters. Each patch gets a feature vector: the average strength of every Gabor response inside it (8 filters, so 8 numbers, in log scale). A single Gaussian (mean and covariance) is fitted to the patches of the good training images, and a test patch is scored by its **Mahalanobis distance** from it, the multi-dimensional version of the z-score that also accounts for correlated features. The image score is the 99.5th percentile of the patch distances. No template is used, so it does not matter where on the leather a pattern appears. This adds a row to the texture table above (run the texture cell first).

In [ ]:
GABOR_KERNELS = [cv2.getGaborKernel((41, 41), 0.56 * lam, np.deg2rad(th), lam, 0.5, 0)
                 for lam in (8, 16) for th in (0, 45, 90, 135)]
GABOR_KERNELS = [k / np.abs(k).sum() for k in GABOR_KERNELS]


def gabor_features(img, patch=16, step=4):
    """Local Gabor energy per patch: one vector of len(GABOR_KERNELS) numbers per position."""
    g = prep(img, 256)
    energy = [cv2.boxFilter(np.abs(cv2.filter2D(g, cv2.CV_32F, k)), -1, (patch, patch)) for k in GABOR_KERNELS]
    return np.log(np.stack(energy, axis=-1)[::step, ::step] + 1e-6)  # log makes the energies more Gaussian


def build_gabor_model(ctg):
    f = np.concatenate([gabor_features(im).reshape(-1, len(GABOR_KERNELS))
                        for im in X[select(ctg, "train", good=True)]])
    cov = np.cov(f.T) + 1e-6 * np.eye(f.shape[1])
    return dict(mu=f.mean(0), P=np.linalg.inv(cov))


def gabor_map(model, img):
    """Mahalanobis distance of every patch from the good patches."""
    f = gabor_features(img)
    d = f.reshape(-1, f.shape[-1]) - model["mu"]
    return np.sqrt(np.einsum("ij,jk,ik->i", d, model["P"], d)).reshape(f.shape[:2])


def make_gabor_score(ctg):
    model = build_gabor_model(ctg)
    return lambda img: np.percentile(gabor_map(model, img), PCTL)


for ctg_t in ["leather", "carpet", "grid"]:
    tex[ctg_t]["Gabor + Mahalanobis"] = evaluate_fn(ctg_t, make_gabor_score(ctg_t))

print(pd.DataFrame({c: {m: r["auroc"] for m, r in d.items()} for c, d in tex.items()}).round(3))


In [ ]:

# anomaly map of one defective leather image
ct = "leather"
gm = build_gabor_model(ct)
i = np.random.choice(select(ct, "test", good=False))
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(prep(X[i], 256), cmap="gray"); ax[0].set_title(ct +f": {labels[i]}")
ax[1].imshow(gabor_map(gm, X[i]), cmap="hot"); ax[1].set_title("Mahalanobis distance of each patch")
for a in ax:
    a.axis("off")
plt.show()

## Deep methods

The classical methods above needed a different hand-made description for every kind of surface. The deep methods replace that description with features from a network pretrained on ImageNet, with no training on defects and no retuning per category.

### Simple baseline: pretrained CNN features + nearest neighbor

The same idea as Gabor + Mahalanobis, with two substitutions:

- the **features** are the layer-2 activations of a pretrained wide-ResNet-50 (a 512-number description of every 8x8 pixel patch at 224 px) instead of Gabor energies
- the **model of normal** is a memory of good-patch features, and a test patch is scored by the distance to its **nearest neighbor** in that memory, instead of a single Gaussian

The image score is the worst patch (the maximum). The memory is a random subset of 40,000 good patches to keep the search fast on a CPU. PatchCore adds a smarter subset (coreset), neighbourhood aggregation of the features, and a better layer combination; this is the version before those refinements.

#### Step by step (for learning)

The next cells build the same method one piece at a time, with a picture at every step. They all use one category (`ctg_w`) and one good and one defective test image.

Four words that come up:

- **Pretrained network**: a neural network that was already trained on over a million everyday photos (ImageNet) to recognise objects. We never train it. We only use its inner layers as a machine that *describes* image content.
- **Feature**: a list of numbers the network produces to describe something about the image.
- **Patch**: a small square region of the image. Here one patch is 8 x 8 pixels (at 224 px), and its description also depends on the area around it.
- **Nearest neighbor**: the most similar item in a collection. Two lists of numbers are "similar" when the distance between them is small.

**Step 0: load the ImageNet model.** Load a model, set it in evaluation mode and check  what is inside.

In [ ]:
import torch
from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights

# disable backpropagation to save memory since we do not train
torch.set_grad_enabled(False)

# load the model in evaluation mode since we will not train it
# batch normalization uses fixed values for mean and std from imagenet training
# dropout passes values without doing anything
# this ensures features are deterministic and reproducible
backbone = wide_resnet50_2(weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1).eval()

# print(backbone)                                   # the full architecture, every layer
print([n for n, _ in backbone.named_children()])    # just the top-level blocks
print(backbone.layer1[0])                         # one block in detail

# get the mean and std values so we can normalize the image set to match model training
Wide_ResNet50_2_Weights.IMAGENET1K_V1.transforms()   # shows resize, crop, mean, std


**Step 1: what does the network see?** Run one image through the first layers of the network. Each layer produces many *feature maps*, one per channel, each a small image showing where that channel responds. Early layers (layer 1) react to simple things like edges and fine texture. Deeper layers (layer 2) react to larger, more complex patterns. The six channels shown per layer are picked at random.

In [ ]:
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

ctg_w = "pill"  # category used in the walkthrough
good_i = np.random.choice(select(ctg_w, "test", good=True))
bad_i = np.random.choice(select(ctg_w, "test", good=False))


def im224(i):
    return cv2.resize(X[i], (224, 224), interpolation=cv2.INTER_AREA)


# numpy image (H, W, C) -->  image tensor (C, H, W)
def to_tensor(imgs):
    x = np.stack([cv2.resize(im, (224, 224), interpolation=cv2.INTER_AREA) for im in imgs])
    return (torch.from_numpy(x).permute(0, 3, 1, 2) - IMNET_MEAN) / IMNET_STD

# build up layer by layer, to look at inner features
def layers(x, upto=3):
    b = backbone
    x = b.maxpool(b.relu(b.bn1(b.conv1(x))))
    out = [b.layer1(x)]
    if upto >= 2:
        out.append(b.layer2(out[0]))
    if upto >= 3:
        out.append(b.layer3(out[1]))
    return out


def feature_map(imgs, layer=2, bs=16):
    """RGB float images -> features of one layer, shape (N, h, w, channels)."""
    out = [layers(to_tensor(imgs[s:s + bs]), upto=layer)[layer - 1].permute(0, 2, 3, 1).numpy()
           for s in range(0, len(imgs), bs)]
    return np.concatenate(out)


l1, l2 = layers(to_tensor([X[bad_i]]), upto=2)
rng_w = np.random.default_rng(0)

fig, ax = plt.subplots(3, 6, figsize=(13, 7))
for a in ax.ravel():
    a.axis("off")
ax[0, 0].imshow(im224(bad_i)); ax[0, 0].set_title(f"{ctg_w}: {labels[bad_i]}")
for row, (fm, name) in enumerate([(l1[0], "layer 1"), (l2[0], "layer 2")], start=1):
    for col, ch in enumerate(rng_w.choice(fm.shape[0], 6, replace=False)):
        ax[row, col].imshow(fm[ch].numpy(), cmap="viridis")
        ax[row, col].set_title(f"{name}, channel {ch}\n{fm.shape[1]}x{fm.shape[2]} pixels", fontsize=9)
plt.tight_layout()
plt.show()

**Step 2: one patch is one list of numbers.** We use layer 2, which has a 28 x 28 grid of positions, so a 224 px image is cut into 784 patches of 8 x 8 pixels. At each position the network gives **512 numbers**. That list is the patch's description (its *feature vector*). The first picture shows the grid and one chosen patch (red), the second a zoomed crop around it (the box is the patch itself), and the third its 512 numbers as bars.

In [ ]:
fm_bad = feature_map([X[bad_i]])[0]   # (28, 28, 512): one 512-number vector per patch
fm_good = feature_map([X[good_i]])[0]
print("features of one image:", fm_bad.shape, "-> 28 x 28 =", 28 * 28, "patches, 512 numbers each")


def crop(im, r, c, ctx=24):
    """Crop around patch (r, c) of a 28x28 grid, with `ctx` extra pixels on each side of the 8x8 patch."""
    pad = cv2.copyMakeBorder(im, ctx, ctx, ctx, ctx, cv2.BORDER_REFLECT)
    y0, x0 = r * 8, c * 8
    return pad[y0:y0 + 8 + 2 * ctx, x0:x0 + 8 + 2 * ctx]


r0, c0 = 14, 14  # the patch we look at
img = im224(bad_i)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(img)
for k in range(0, 225, 8):
    ax[0].axhline(k - 0.5, color="w", lw=0.3, alpha=0.5); ax[0].axvline(k - 0.5, color="w", lw=0.3, alpha=0.5)
ax[0].add_patch(plt.Rectangle((c0 * 8 - 0.5, r0 * 8 - 0.5), 8, 8, fill=False, ec="red", lw=2))
ax[0].set_title("28 x 28 grid of patches (8 x 8 px each)")
ax[1].imshow(crop(img, r0, c0))
ax[1].add_patch(plt.Rectangle((23.5, 23.5), 8, 8, fill=False, ec="red", lw=2))
ax[1].set_title("the chosen patch, zoomed")
ax[2].bar(range(512), fm_bad[r0, c0], width=1.0)
ax[2].set_title("its description: 512 numbers"); ax[2].set_xlabel("channel")
for a in ax[:2]:
    a.axis("off")
plt.tight_layout()
plt.show()

**Step 3: similar patches get similar numbers.** Pick a patch and compute the *distance* between its 512 numbers and those of every other patch in the image (the usual straight-line distance, as between two points, only in 512 dimensions). Dark means a small distance, so a similar description. If the descriptions capture what the surface looks like, patches of the same kind should be dark together. This is why distances between descriptions are a meaningful way to compare patches, and not just a comparison of raw pixels.

In [ ]:
pts = {"A": (2, 2), "B": (14, 14)}  # (row, col) of two patches to compare against; edit these
colors = {"A": "tab:blue", "B": "tab:orange"}

fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
ax[0].imshow(im224(bad_i)); ax[0].set_title("two chosen patches")
for n, (r, c) in pts.items():
    ax[0].add_patch(plt.Rectangle((c * 8 - 2, r * 8 - 2), 12, 12, fill=False, ec=colors[n], lw=2))
    d = np.linalg.norm(fm_bad - fm_bad[r, c], axis=-1)
    a = ax[1] if n == "A" else ax[2]
    a.imshow(d, cmap="viridis"); a.set_title(f"distance to patch {n}  (dark = similar)", color=colors[n])
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

**Step 4: the memory of normal.** Now do the same for all the **good training images** and pile every patch description into one big collection, the *memory*. Each image gives 784 patches, so the memory holds a few hundred thousand lists of 512 numbers. (Here a random 40,000 of them are kept, to make the search fast.)

To see it, squash each 512-number list to two numbers with PCA (a way to keep the two directions in which the patches differ most) and plot them. Grey dots are good training patches, blue dots the patches of a good *test* image, and red dots the patches of a *defective* test image.

The three colours overlap heavily. That is expected, and is itself the lesson: in two dimensions almost everything looks alike, because the differences between a normal and a defective patch live in the other 510 directions. This is why the method measures distance in the full 512-number space, as the next step does.

In [ ]:
from sklearn.decomposition import PCA

train_imgs = list(X[select(ctg_w, "train", good=True)])
train_f = feature_map(train_imgs).reshape(-1, 512)          # every patch of every good training image
keep = np.random.default_rng(0).choice(len(train_f), size=min(40000, len(train_f)), replace=False)
memory = torch.from_numpy(train_f[keep])                     # the memory of normal
print(f"{len(train_imgs)} training images x 784 patches = {len(train_f):,} patches, memory keeps {len(keep):,}")

#reduce to 2 principal components
pca = PCA(n_components=2).fit(train_f[keep])
p_mem = pca.transform(train_f[keep][:6000])
p_good = pca.transform(fm_good.reshape(-1, 512))
p_bad = pca.transform(fm_bad.reshape(-1, 512))

plt.figure(figsize=(7, 6))
plt.scatter(*p_mem.T, s=3, c="lightgrey", label="good training patches (memory)")
plt.scatter(*p_good.T, s=8, c="tab:blue", label="patches of a good test image")
plt.scatter(*p_bad.T, s=8, c="tab:red", label="patches of the defective test image")
plt.legend(loc="best", fontsize=8); plt.xlabel("PCA 1"); plt.ylabel("PCA 2")
plt.title("each dot is one patch (512 numbers squashed to 2)")
plt.show()

**Step 5: nearest neighbor for one patch.** For a test patch, search the memory for the most similar good patch and take that distance as its anomaly score. Below, the patch with the **largest** distance (the most suspicious one) and a **typical** patch (median distance) are each shown next to their nearest match from the good images. A normal patch finds a near-identical twin. A defective patch has no good lookalike, so even its closest match is clearly different, and that gap is the whole method.

The histogram underneath shows the nearest-neighbor distance of **every** patch of a good and of the defective test image. Most patches of both images are close to something in the memory; the difference is in the tail of far-away patches.

In [ ]:
q = torch.from_numpy(fm_bad.reshape(-1, 512))
dist, nn = torch.cdist(q, memory).min(dim=1)       # per test patch: distance to, and index of, its nearest good patch
dist, nn = dist.numpy(), nn.numpy()

worst, typical = int(dist.argmax()), int(np.argsort(dist)[len(dist) // 2])
img = im224(bad_i)

fig = plt.figure(figsize=(13, 5))
gs = fig.add_gridspec(2, 3, width_ratios=[1.6, 1, 1])
axm = fig.add_subplot(gs[:, 0]); axm.imshow(img); axm.axis("off"); axm.set_title("test image")
for row, (name, p, col) in enumerate([("most suspicious", worst, "red"), ("typical", typical, "limegreen")]):
    r, c = divmod(p, 28)
    axm.add_patch(plt.Rectangle((c * 8 - 2, r * 8 - 2), 12, 12, fill=False, ec=col, lw=2))
    g = keep[nn[p]]                                   # which training image and position the nearest match came from
    gi, gr, gc = g // 784, (g % 784) // 28, (g % 784) % 28
    a1, a2 = fig.add_subplot(gs[row, 1]), fig.add_subplot(gs[row, 2])
    a1.imshow(crop(img, r, c)); a1.set_title(f"{name} patch", color=col)
    a2.imshow(crop(cv2.resize(train_imgs[gi], (224, 224), interpolation=cv2.INTER_AREA), gr, gc))
    a2.set_title(f"nearest good patch\ndistance {dist[p]:.1f}")
    a1.axis("off"); a2.axis("off")
plt.tight_layout()
plt.show()

dist_good = torch.cdist(torch.from_numpy(fm_good.reshape(-1, 512)), memory).min(dim=1).values.numpy()
plt.figure(figsize=(7, 3.5))
bins = np.linspace(0, max(dist.max(), dist_good.max()), 40)
plt.hist(dist_good, bins, alpha=0.6, label=f"good image (max {dist_good.max():.1f})")
plt.hist(dist, bins, alpha=0.6, label=f"defective image (max {dist.max():.1f})")
plt.xlabel("distance of a patch to its nearest good patch"); plt.ylabel("patches"); plt.legend()
plt.show()


**Step 6: from patch distances to an image score.** Do the nearest-neighbor search for *every* patch, and the 28 x 28 distances form an **anomaly map**, which is shown here over the image after upsampling and a little smoothing. The **image score** is simply the largest distance in the map: one badly unfamiliar patch is enough to make the whole part suspicious. Good and defective image are on the same colour scale, so they can be compared.

In [ ]:
def knn_map_w(img):
    """28x28 map: distance of every patch to its nearest good patch."""
    q = torch.from_numpy(feature_map([img])[0].reshape(-1, 512))
    return torch.cdist(q, memory).min(dim=1).values.reshape(28, 28).numpy()


maps = {i: knn_map_w(X[i]) for i in (good_i, bad_i)}
vmax = max(m.max() for m in maps.values())

fig, ax = plt.subplots(2, 2, figsize=(9, 9))
for col, (i, m) in enumerate(maps.items()):
    up = cv2.GaussianBlur(cv2.resize(m, (224, 224), interpolation=cv2.INTER_LINEAR), (0, 0), 4)
    ax[0, col].imshow(im224(i)); ax[0, col].set_title(f"{labels[i]}")
    ax[1, col].imshow(im224(i)); ax[1, col].imshow(up, cmap="jet", alpha=0.5, vmin=0, vmax=vmax)
    ax[1, col].set_title(f"image score = max = {m.max():.1f}")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

**Step 7: check it on all test images.** Score every test image of the category and compare the scores of good and defective images, as with the classical methods. The left plot shows the two score distributions (how well they separate), the right plot the ROC curve and its AUROC.

In [ ]:
from sklearn.metrics import roc_curve

r_w = evaluate_fn(ctg_w, lambda img: knn_map_w(img).max())
print(f"{ctg_w}: image AUROC = {r_w['auroc']:.3f}   {r_w['s_per_img']:.2f} s/img")
print({k: round(v, 2) for k, v in r_w["per_defect"].items()})

idx_w = select(ctg_w, "test")
y_w = (labels[idx_w] != "good").astype(int)
s_w = np.array([knn_map_w(X[i]).max() for i in idx_w])
fpr, tpr, _ = roc_curve(y_w, s_w)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.linspace(s_w.min(), s_w.max(), 30)
ax[0].hist(s_w[y_w == 0], bins, alpha=0.6, label="good"); ax[0].hist(s_w[y_w == 1], bins, alpha=0.6, label="defective")
ax[0].set_xlabel("image score"); ax[0].set_ylabel("images"); ax[0].legend()
ax[1].plot(fpr, tpr, label=f"AUROC = {r_w['auroc']:.3f}"); ax[1].plot([0, 1], [0, 1], "k--")
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate"); ax[1].legend()
plt.tight_layout()
plt.show()

**Optional experiments: change one thing and see what happens.**

- **Which layer?** Layer 1 describes small, simple structures (a 56 x 56 grid, 256 numbers). Layer 3 describes larger, more abstract ones (a 14 x 14 grid, 1024 numbers).
- **How big is the memory?** Fewer stored patches means a faster search, but a normal patch may no longer find a good twin, which creates false alarms.

Each row below builds the model and scores all test images of `ctg_w`, so this takes a few minutes. Edit the list to try other combinations.

In [ ]:
def knn_variant(ctg, layer, max_patches):
    f = feature_map(list(X[select(ctg, "train", good=True)]), layer)
    ch = f.shape[-1]
    f = f.reshape(-1, ch)
    keep_v = np.random.default_rng(0).choice(len(f), size=min(max_patches, len(f)), replace=False)
    mem = torch.from_numpy(f[keep_v])
    return evaluate_fn(ctg, lambda img: torch.cdist(torch.from_numpy(feature_map([img], layer)[0].reshape(-1, ch)), mem).min(dim=1).values.max().item())


variants = [(1, 40000), (2, 5000), (2, 40000), (3, 40000)]  # (layer, memory size)
rows = []
for layer, n in variants:
    r_v = knn_variant(ctg_w, layer, n)
    rows.append({"layer": layer, "memory patches": n, "AUROC": round(r_v["auroc"], 3), "s/img": round(r_v["s_per_img"], 2)})
    print(rows[-1], flush=True)
pd.DataFrame(rows)

#### Where does it fail? Look at the mistakes

An AUROC says how well good and defective images are separated, but not *why* some are not. The function below works for any method (classical or deep): give it a score function (one number per image) and a map function (the anomaly map). It sets the threshold so that at most `TARGET_FPR` of the good test images are rejected, then shows the **good images with the highest scores** (what the threshold has to get past) and the **defective images with the lowest scores** (the ones that slip through), on one common colour scale. It also prints which defect types are missed.

Two things to look for: is the heat of a missed defect simply weaker than the normal variation of the good images (a threshold problem), or does it sit somewhere the method cannot see (a feature problem)? And are the false alarms caused by the object itself, or by artefacts such as borders, lighting or a slightly shifted part?

In [ ]:
def show_failures(ctg, score_fn, map_fn, k=3, target_fpr=TARGET_FPR):
    """The k hardest good images and the k most clearly missed defects of any method.

    score_fn(img) -> one number per image, map_fn(img) -> 2D anomaly map.
    The threshold lets at most `target_fpr` of the good test images through as false alarms.
    """
    idx = select(ctg, "test")
    y = (labels[idx] != "good").astype(int)
    s = np.array([score_fn(X[i]) for i in idx])
    thr = np.quantile(s[y == 0], 1 - target_fpr)

    missed = (y == 1) & (s <= thr)
    print(f"{ctg}: threshold {thr:.2f}, misses {missed.sum()} of {y.sum()} defective images")
    print(pd.Series(labels[idx][missed]).value_counts().to_dict() or "no misses")

    good_hard = np.where(y == 0)[0][np.argsort(-s[y == 0])[:k]]   # good images with the highest scores
    def_missed = np.where(y == 1)[0][np.argsort(s[y == 1])[:k]]   # defective images with the lowest scores
    rows = [("hardest good image", good_hard), ("weakest defective image", def_missed)]
    maps = {j: map_fn(X[idx[j]]) for _, js in rows for j in js}
    vmax = max(m.max() for m in maps.values())                    # one colour scale, so all are comparable

    fig, ax = plt.subplots(2, k, figsize=(4 * k, 8.5))
    for r_i, (name, js) in enumerate(rows):
        for c_i, j in enumerate(js):
            i = idx[j]
            up = cv2.resize(maps[j], X[i].shape[1::-1], interpolation=cv2.INTER_LINEAR)
            ax[r_i, c_i].imshow(X[i]); ax[r_i, c_i].imshow(up, cmap="jet", alpha=0.45, vmin=0, vmax=vmax)
            verdict = "rejected" if s[j] > thr else "passed"
            ax[r_i, c_i].set_title(f"{name}\n{labels[i]}, score {s[j]:.2f} ({verdict})", fontsize=9)
    for a in ax.ravel():
        a.axis("off")
    plt.tight_layout()
    plt.show()


show_failures(ctg_w, lambda img: knn_map_w(img).max(), knn_map_w)

#### The whole baseline in one cell, for all categories

The steps above, put together as one function per category: build the memory of good patches, score every test image by its worst patch, and evaluate. This is the version used for the results table.

In [ ]:
import torch
from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights

torch.set_grad_enabled(False)
torch.manual_seed(0)
backbone = wide_resnet50_2(weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1).eval()
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def cnn_features(imgs, bs=16):
    """RGB float images -> layer-2 feature map per image, shape (N, 28, 28, 512)."""
    out = []
    for s in range(0, len(imgs), bs):
        x = np.stack([cv2.resize(im, (224, 224), interpolation=cv2.INTER_AREA) for im in imgs[s:s + bs]])
        x = (torch.from_numpy(x).permute(0, 3, 1, 2) - IMNET_MEAN) / IMNET_STD
        b = backbone
        x = b.layer2(b.layer1(b.maxpool(b.relu(b.bn1(b.conv1(x))))))
        out.append(x.permute(0, 2, 3, 1).numpy())
    return np.concatenate(out)


def build_knn_memory(ctg, max_patches=40000, seed=0):
    """Memory bank: patch features of the good training images (random subset, to keep the search fast)."""
    f = cnn_features(list(X[select(ctg, "train", good=True)])).reshape(-1, 512)
    rng = np.random.default_rng(seed)
    keep = rng.choice(len(f), size=min(max_patches, len(f)), replace=False)
    return torch.from_numpy(f[keep])


def knn_map(memory, img):
    """Distance of every patch to its nearest good patch."""
    q = torch.from_numpy(cnn_features([img])[0].reshape(-1, 512))
    return torch.cdist(q, memory).min(dim=1).values.reshape(28, 28).numpy()


def make_knn_score(ctg):
    memory = build_knn_memory(ctg)
    return lambda img: knn_map(memory, img).max()  # worst patch = image score


deep = {}
for ctg_d in CATEGORIES:
    t0 = time.perf_counter()
    fn = make_knn_score(ctg_d)
    build_s = time.perf_counter() - t0
    deep[ctg_d] = evaluate_fn(ctg_d, fn)
    deep[ctg_d]["build_s"] = build_s
    print(f"{ctg_d:12s} AUROC {deep[ctg_d]['auroc']:.3f}   {deep[ctg_d]['s_per_img']:.2f} s/img   (build {build_s:.0f} s)")

### Improving the baseline

The failure view above points at two things worth trying on the same nearest-neighbor method: leaving out the image border, and a different backbone.

#### Side experiment: ignore the border

The failure plot of `grid` showed that the highest scores of good images sit at the edges and corners. A likely cause is the zero padding inside the network, which makes the features at the image edge unlike anything in the memory. If so, the outermost patches should be left out when taking the maximum.

How many to leave out is decided **without any test image**: a fifth of the good training images is held out, the memory is built from the rest, and the held-out patches are compared with it ring by ring from the border. The margin is the first ring whose distance is within 5% of the interior level (`tol`), but at most 10% of the grid size (`max_frac`). The test images are scored only once, with that margin. For comparison the table also shows the plain baseline.

What to expect: where the plot is flat (objects on a plain background) the margin is 0 and nothing changes. Where the border is clearly higher, the margin is larger and the AUROC can improve. The 5% limit is a choice, and a border margin could hide a defect that sits at the very edge of the image.

In [ ]:
def ring_profile(maps):
    """Mean distance of the held-out good images per ring from the border (0 = outermost ring)."""
    g = maps.shape[1]
    prof = []
    for r in range(g // 2):
        mask = np.zeros((g, g), bool)
        mask[r:g - r, r:g - r] = True
        if r + 1 < g // 2:
            mask[r + 1:g - r - 1, r + 1:g - r - 1] = False
        prof.append(maps[:, mask].mean())
    return np.array(prof)


def choose_margin(ctg, tol=1.05, max_frac=0.1, seed=0):
    """Border margin (in patches) from good training images only; no test image is used.

    Hold out a fifth of the good training images, build the memory from the rest, and look at how far
    the held-out patches are from their nearest memory patch, ring by ring from the border. The margin
    is the first ring that is within `tol` of the interior level, but at most `max_frac` of the grid size
    (so a coarse grid cannot lose a large part of the image).
    """
    tr = select(ctg, "train", good=True)
    perm = np.random.default_rng(seed).permutation(len(tr))
    n_val = len(tr) // 5
    val_i, fit_i = tr[perm[:n_val]], tr[perm[n_val:]]
    f = cnn_features(list(X[fit_i])).reshape(-1, 512)
    keep = np.random.default_rng(seed).choice(len(f), size=min(40000, len(f)), replace=False)
    memory = torch.from_numpy(f[keep])
    maps = np.array([knn_map(memory, X[i]) for i in val_i])
    prof = ring_profile(maps)
    interior = np.median(prof[len(prof) // 2:])
    margin = int(np.argmax(prof <= tol * interior))
    return min(margin, int(max_frac * maps.shape[1])), prof / interior


rows, profiles = [], {}
for ctg_m in CATEGORIES:
    margin, profiles[ctg_m] = choose_margin(ctg_m)
    memory = build_knn_memory(ctg_m)              # same memory as the baseline above
    idx = select(ctg_m, "test")
    y = (labels[idx] != "good").astype(int)
    maps = np.array([knn_map(memory, X[i]) for i in idx])
    g = maps.shape[1]
    plain = roc_auc_score(y, maps.reshape(len(idx), -1).max(1))
    cropped = roc_auc_score(y, maps[:, margin:g - margin, margin:g - margin].reshape(len(idx), -1).max(1))
    rows.append({"category": ctg_m, "margin": margin, "AUROC plain": round(plain, 3), "AUROC ignoring border": round(cropped, 3)})
    print(rows[-1], flush=True)

plt.figure(figsize=(7, 3.5))
for ctg_m, prof in profiles.items():
    plt.plot(prof[:8], marker="o", label=ctg_m)
plt.axhline(1.05, color="k", ls="--", lw=1)
plt.xlabel("ring from the border (0 = outermost)"); plt.ylabel("distance / interior level")
plt.title("how much the border inflates the score (held-out good images)"); plt.legend(fontsize=8)
plt.show()
border_df = pd.DataFrame(rows)
border_df

**Does ignoring the border hide defects at the edge?** Check with the ground-truth masks, no model needed: for each category, how many defective test images have defect pixels inside the ignored border ring, and how many have *all* of their defect pixels there (those cannot be found at all with this margin). The ring is `margin` patches wide on the 28 x 28 feature grid, which is `margin * 512 / 28` pixels.

What to look at:

- The margin hides very little. Defect images with pixels in the ring: leather 8 of 92, carpet 12 of 89, grid 22 of 57. The share of defect pixels in the ring is 1%, 3% and 10%. The three categories with margin 0 lose nothing.
- Only one image (grid) has its whole defect in the ring, so it is lost for sure with this margin. The other images are only partly affected: the feature patches reach across the cut, so they are not necessarily missed.
- What the scores do (grid, WRN50 kNN, threshold at the 95th percentile of the good test scores, so optimistic): of the 22 images with pixels in the ring, 5 are caught without the margin and 21 with it; none that is caught without the margin is lost by it. The one miss is the image whose whole defect is in the ring (`glue/008`), which is also missed without the margin. So on grid the margin costs one image and gains far more, but it is a real blind spot: a defect at the very edge cannot be found.
- The figure below shows the border-only image and the two partly affected images with the largest share of the defect in the ring.

In [ ]:
G = 28  # feature grid of the WRN50 stage used for the border margin
rows = []
for _, r in border_df.iterrows():
    b = round(r["margin"] * RESOLUTION / G)
    n_any = n_only = 0
    share = []
    for f in sorted((PATH_ROOT / r["category"] / "ground_truth").glob("*/*_mask.png")):
        m = cv2.resize(cv2.imread(str(f), 0), (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_NEAREST) > 0
        inner = np.zeros_like(m)
        inner[b:RESOLUTION - b, b:RESOLUTION - b] = True
        n_any += (m & ~inner).any()
        n_only += not (m & inner).any()
        share.append((m & ~inner).sum() / m.sum())
    rows.append({"category": r["category"], "margin px": b, "defect images": len(share), "pixels in ring": int(n_any),
                 "only in ring": int(n_only), "share of defect pixels in ring": round(float(np.mean(share)), 3)})
pd.DataFrame(rows)

In [ ]:
ctg_e = "grid"
b = round(border_df.set_index("category").loc[ctg_e, "margin"] * RESOLUTION / G)
cands = []
for f in sorted((PATH_ROOT / ctg_e / "ground_truth").glob("*/*_mask.png")):
    m = cv2.resize(cv2.imread(str(f), 0), (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_NEAREST) > 0
    inner = np.zeros_like(m)
    inner[b:RESOLUTION - b, b:RESOLUTION - b] = True
    cands.append((f, m, (m & ~inner).sum() / m.sum()))
only = [c for c in cands if c[2] == 1.0]
partly = sorted([c for c in cands if 0 < c[2] < 1], key=lambda c: -c[2])[:2]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (f, m, share) in zip(axes, only + partly):
    img = cv2.cvtColor(cv2.imread(str(f.parents[2] / "test" / f.parent.name / f.name.replace("_mask", ""))), cv2.COLOR_BGR2RGB)
    ax.imshow(cv2.resize(img, (RESOLUTION, RESOLUTION)))
    for c in cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        ax.plot(*np.r_[c[:, 0, :], c[:1, 0, :]].T, color="red", lw=1.5)
    ax.add_patch(plt.Rectangle((b, b), RESOLUTION - 2 * b, RESOLUTION - 2 * b, fill=False, ec="cyan", ls="--", lw=1.5))
    ax.set_title(f"{f.parent.name}/{f.stem.replace('_mask', '')}: {share:.0%} of the defect in the ring", fontsize=9)
    ax.axis("off")
plt.suptitle("grid: red = defect, dashed = inner edge of the ignored border ring", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
def border_scores(maps, margin):
    g = maps.shape[1]
    return maps[:, margin:g - margin, margin:g - margin].reshape(len(maps), -1).max(1)


# grid: do the scores miss the images whose defect touches the ignored ring? (threshold: 95th percentile of the good test scores, so optimistic)
ctg_x = "grid"
mg = int(border_df.set_index("category").loc[ctg_x, "margin"])
b = round(mg * RESOLUTION / G)
memory = build_knn_memory(ctg_x)
files = sorted((PATH_ROOT / ctg_x / "test").glob("*/*.png"))
maps = np.array([knn_map(memory, cv2.resize(cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB), (RESOLUTION, RESOLUTION)).astype("float32") / 255.0)
                 for f in files])
y = np.array([f.parent.name != "good" for f in files])
s_plain, s_crop = border_scores(maps, 0), border_scores(maps, mg)
thr_plain, thr_crop = np.quantile(s_plain[~y], 0.95), np.quantile(s_crop[~y], 0.95)

touch, only = [], []  # defect images with pixels in the ring / with all defect pixels in the ring
for i, f in enumerate(files):
    if not y[i]:
        continue
    m = cv2.resize(cv2.imread(str(PATH_ROOT / ctg_x / "ground_truth" / f.parent.name / f"{f.stem}_mask.png"), 0),
                   (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_NEAREST) > 0
    inner = np.zeros_like(m)
    inner[b:RESOLUTION - b, b:RESOLUTION - b] = True
    if (m & ~inner).any():
        touch.append(i)
    if not (m & inner).any():
        only.append(i)
touch, only = np.array(touch), np.array(only)
print(f"AUROC plain {roc_auc_score(y, s_plain):.3f}, ignoring border {roc_auc_score(y, s_crop):.3f}")
print(f"defect images touching the ring: {len(touch)}; caught without margin {(s_plain[touch] > thr_plain).sum()}, with margin {(s_crop[touch] > thr_crop).sum()}")
print(f"caught without the margin but lost with it: {((s_plain[touch] > thr_plain) & (s_crop[touch] <= thr_crop)).sum()}")
print("defect only in the ring:", [(files[i].parent.name + "/" + files[i].stem, bool(s_plain[i] > thr_plain), bool(s_crop[i] > thr_crop)) for i in only])

#### Side experiment: another backbone (ConvNeXt)

Only the feature extractor changes, everything else is as in the baseline: patch features of the good training images as memory, nearest-neighbor distance, image score = worst patch, and the border margin chosen from held-out good images. ConvNeXt-tiny is a modern network with the same four-stage structure as the ResNet. Its stage 2 gives the same 28 x 28 grid as the layer 2 used above, its stage 3 a coarser 14 x 14 grid.

How to read the table:

- The best stage depends on the category. The coarser stage 3 helps the parts (`transistor`, `metal_nut`), the finer stage 2 is needed for `grid`, where the defects are small. `pill` hardly changes with any setting, so the backbone is not what limits it there.
- The border margin is capped at 10% of the grid size, which is one patch on the coarse 14 x 14 grid. Without the cap, an earlier run picked a margin of 4 for `carpet` with stage 3, which throws away a third of each side and lowered its AUROC from 1.000 to 0.941. The cap also limits what the border can do for `grid` with stage 3.
- One random seed and small test sets: differences of 0.01 to 0.02 are within noise. Picking the best stage per category from these numbers would be tuning on the test set.
- Speed is compared for all methods in one place further below.

In [ ]:
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

convnext = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1).eval()  # same ImageNet mean and std as above


def convnext_features(imgs, stage, bs=16):
    """RGB float images -> features after convnext.features[stage], shape (N, h, w, channels)."""
    out = []
    for s in range(0, len(imgs), bs):
        x = np.stack([cv2.resize(im, (224, 224), interpolation=cv2.INTER_AREA) for im in imgs[s:s + bs]])
        x = (torch.from_numpy(x).permute(0, 3, 1, 2) - IMNET_MEAN) / IMNET_STD
        for k in range(stage + 1):
            x = convnext.features[k](x)
        out.append(x.permute(0, 2, 3, 1).numpy())
    return np.concatenate(out)


def memory_of(idxs, stage, max_patches=40000, seed=0):
    f = convnext_features(list(X[idxs]), stage)
    ch = f.shape[-1]
    f = f.reshape(-1, ch)
    keep = np.random.default_rng(seed).choice(len(f), size=min(max_patches, len(f)), replace=False)
    return torch.from_numpy(f[keep]), ch


def knn_maps(idxs, memory, ch, stage):
    out = []
    for i in idxs:
        fm = convnext_features([X[i]], stage)[0]
        h, w = fm.shape[:2]
        out.append(torch.cdist(torch.from_numpy(fm.reshape(-1, ch)), memory).min(dim=1).values.reshape(h, w).numpy())
    return np.array(out)


def choose_margin_convnext(ctg, stage, tol=1.05, max_frac=0.1, seed=0):
    """Same held-out procedure as above: border margin from good training images only, capped at max_frac of the grid."""
    tr = select(ctg, "train", good=True)
    perm = np.random.default_rng(seed).permutation(len(tr))
    n_val = len(tr) // 5
    memory, ch = memory_of(tr[perm[n_val:]], stage)
    maps = knn_maps(tr[perm[:n_val]], memory, ch, stage)
    prof = ring_profile(maps)
    margin = int(np.argmax(prof <= tol * np.median(prof[len(prof) // 2:])))
    return min(margin, int(max_frac * maps.shape[1]))


rows = []
for stage, name in [(3, "stage 2 (28x28, 192 ch)"), (5, "stage 3 (14x14, 384 ch)")]:
    for ctg_c in CATEGORIES:
        margin = choose_margin_convnext(ctg_c, stage)
        memory, ch = memory_of(select(ctg_c, "train", good=True), stage)
        idx = select(ctg_c, "test")
        y = (labels[idx] != "good").astype(int)
        maps = knn_maps(idx, memory, ch, stage)
        g = maps.shape[1]
        plain = roc_auc_score(y, maps.reshape(len(idx), -1).max(1))
        cropped = roc_auc_score(y, maps[:, margin:g - margin, margin:g - margin].reshape(len(idx), -1).max(1))
        rows.append({"features": name, "category": ctg_c, "margin": margin, "AUROC plain": round(plain, 3),
                     "AUROC ignoring border": round(cropped, 3)})
        print(rows[-1], flush=True)
convnext_df = pd.DataFrame(rows)
convnext_df

### PatchCore with anomalib

#### The idea behind the coreset, and what else PatchCore changes

The memory of good patches is big: every good training image gives 784 patches, a few hundred thousand in total, and every test patch is compared with all of them. So some of them have to be dropped. The simple baseline keeps a **random** subset. A random subset follows the *density* of the patches: the common kinds (plain background) are kept many times over, while a rare but normal kind may be missing altogether. A normal patch of that kind then has no close match in the memory and gets a high score, which is a false alarm.

A **coreset** is chosen to *cover* instead. The greedy k-center rule is simple: start with one patch, then keep adding the patch that is farthest from all patches chosen so far. After k steps, every patch of the full set has a chosen patch not too far away, the rare ones included.

The figure compares both with the same 1000 patches of `pill` (0.6% of all its patches; PatchCore keeps 10%, here the memory is made much smaller so that the difference is visible). Left and middle: where the patches sit, projected to two dimensions. The random subset follows the dense clusters, the coreset also reaches the thin regions. Right: the distance of *held-out good* patches to the closest memory patch. The coreset has fewer patches that are close to the memory, because it spends points on the rare ones, but a shorter tail: the worst-covered normal patch is closer to the memory, and the printed test AUROC of the same search is much higher. This is one random draw, and a projection of 512 numbers to 2 only hints at the coverage.

What PatchCore changes compared with the simple baseline:

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead><tr>
<th style="text-align: left;"></th>
<th style="text-align: left;">simple baseline</th>
<th style="text-align: left;">PatchCore</th>
</tr></thead>
<tbody>
<tr><td style="text-align: left;">features</td><td style="text-align: left;">one layer (layer 2)</td><td style="text-align: left;">layers 2 and 3 combined</td></tr>
<tr><td style="text-align: left;">neighbourhood</td><td style="text-align: left;">each patch on its own</td><td style="text-align: left;">features of neighbouring patches are averaged</td></tr>
<tr><td style="text-align: left;">memory</td><td style="text-align: left;">random subset (40,000 patches)</td><td style="text-align: left;">coreset (10% of all patches)</td></tr>
<tr><td style="text-align: left;">score</td><td style="text-align: left;">distance of the worst test patch to its nearest memory patch</td><td style="text-align: left;">the same idea</td></tr>
</tbody>
</table>

The price of the coreset is time: choosing it is a loop over all patches for every kept patch, which is what makes fitting take minutes on the CPU.

In [ ]:
from sklearn.decomposition import PCA

ctg_c = "pill"
k = 1000  # memory size for the comparison: 0.5% of all patches, small enough to see the difference
tr = select(ctg_c, "train", good=True)
perm = np.random.default_rng(0).permutation(len(tr))
val_i, fit_i = tr[perm[:len(tr) // 5]], tr[perm[len(tr) // 5:]]  # held-out good images to check the coverage

f_fit = cnn_features(list(X[fit_i])).reshape(-1, 512)  # every patch of the good training images
f_val = cnn_features(list(X[val_i])).reshape(-1, 512)
all_patches = torch.from_numpy(f_fit)


def coreset(points, k, dim=128, seed=0):
    """Greedy k-center: start from a random patch, then keep adding the patch that is farthest from all chosen ones.
    The distances are computed in a random 128-dim projection to keep it fast."""
    g = torch.Generator().manual_seed(seed)
    z = points @ (torch.randn(points.shape[1], dim, generator=g) / dim ** 0.5)
    chosen = [int(torch.randint(len(z), (1,), generator=g))]
    d = torch.cdist(z, z[chosen[-1]][None]).squeeze(1)
    for _ in range(k - 1):
        chosen.append(int(d.argmax()))
        d = torch.minimum(d, torch.cdist(z, z[chosen[-1]][None]).squeeze(1))
    return np.array(chosen)


random_idx = np.random.default_rng(0).choice(len(all_patches), size=k, replace=False)
core_idx = coreset(all_patches, k)
mem = {"random subset": all_patches[random_idx], "coreset": all_patches[core_idx]}

# how far is a held-out good patch from the closest memory patch? large values would become false alarms
v = torch.from_numpy(f_val)
dist = {n: torch.cdist(v, m).min(dim=1).values.numpy() for n, m in mem.items()}

# the same memories on the test images of the category
idx = select(ctg_c, "test")
y = (labels[idx] != "good").astype(int)
auroc = {n: roc_auc_score(y, [torch.cdist(torch.from_numpy(cnn_features([X[i]]).reshape(-1, 512)), m).min(dim=1).values.max().item() for i in idx])
         for n, m in mem.items()}

pca = PCA(n_components=2).fit(f_fit[np.random.default_rng(0).choice(len(f_fit), 20000, replace=False)])
bg = pca.transform(f_fit[np.random.default_rng(1).choice(len(f_fit), 20000, replace=False)])
colors = {"random subset": "tab:blue", "coreset": "tab:red"}

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
for a, n in zip(ax[:2], mem):
    a.scatter(*bg.T, s=2, c="lightgrey")
    a.scatter(*pca.transform(mem[n].numpy()).T, s=8, c=colors[n])
    a.set_title(f"{n}: {k} of {len(all_patches):,} patches"); a.set_xlabel("PCA 1"); a.set_ylabel("PCA 2")
lim = np.percentile(np.r_[dist["random subset"], dist["coreset"]], 99.9)
bins = np.linspace(0, lim, 50)
for n, d_ in dist.items():
    ax[2].hist(d_, bins, alpha=0.6, color=colors[n], label=n)
ax[2].set_yscale("log"); ax[2].legend()
ax[2].set_xlabel("distance of a held-out good patch to its nearest memory patch"); ax[2].set_ylabel("patches (log scale)")
ax[2].set_title("how well does the memory cover normal patches?")
plt.tight_layout()
plt.show()
for n in mem:
    print(f"{n:14s} worst-covered good patch: {dist[n].max():.2f}, 99th percentile {np.percentile(dist[n], 99):.2f}, test AUROC {auroc[n]:.3f}")

#### PatchCore: all categories

PatchCore is the same idea as our baseline: a memory of good patch features from a pretrained Wide ResNet50, and the distance to the nearest memory patch as the score. It adds three things: the features of layers 2 and 3 are combined, the features of neighbouring patches are averaged, and the memory is a *coreset* (10% of all good patches, chosen so that they cover the whole set as evenly as possible) instead of a random subset.

The cell fits it with anomalib's defaults on the CPU and stores every result in `patchcore_results.csv`. Finished categories are read from that file, only missing ones are run (about 7 to 12 minutes each), so delete the file to repeat everything.

This is not exactly like-for-like with the runs above: anomalib loads and resizes the images itself (256 px, no crop; we used 224 px) and has its own scoring details. The coreset is chosen at random, so a rerun changes the result a little (about 0.003 AUROC on `transistor` between two runs). The per-image scores that the intervals and the false-reject analysis further down need come from a separate fit on 80% of the good images (next cell).

In [ ]:
import warnings

# deprecation warnings that anomalib prints when it is imported; they do not affect the results
for msg in ["Importing from timm.models.layers is deprecated", "The anomalib.models.components.dinov2 package is deprecated", "`torch.jit.script` is not supported"]:
    warnings.filterwarnings("ignore", message=msg, category=FutureWarning)

from anomalib.data import MVTecAD
from anomalib.engine import Engine
from anomalib.models import Patchcore

PATCHCORE_CSV = Path("patchcore_results.csv")


def run_patchcore(ctg):
    """Fit PatchCore (anomalib defaults) on the good training images of one category and test it. Slow on the CPU."""
    dm = MVTecAD(root=PATH_ROOT, category=ctg, train_batch_size=16, eval_batch_size=16, num_workers=0)
    model = Patchcore()
    engine = Engine(accelerator="cpu", default_root_dir="anomalib_runs", logger=False, enable_progress_bar=False)
    t0 = time.perf_counter()
    engine.fit(model=model, datamodule=dm)
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    res = engine.test(model=model, datamodule=dm)[0]
    test_s = time.perf_counter() - t0
    return {"category": ctg, **{k: round(float(v), 4) for k, v in res.items()}, "fit_s": round(fit_s),
            "test_s_per_img": round(test_s / len(dm.test_data), 3), "n_train": len(dm.train_data), "n_test": len(dm.test_data)}


# finished categories are read from the file, only the missing ones are run (about 10 minutes each on the CPU)
pc = pd.read_csv(PATCHCORE_CSV) if PATCHCORE_CSV.exists() else pd.DataFrame(columns=["category"])
for ctg_p in CATEGORIES:
    if ctg_p not in set(pc["category"]):
        pc = pd.concat([pc, pd.DataFrame([run_patchcore(ctg_p)])], ignore_index=True)
        pc.to_csv(PATCHCORE_CSV, index=False)
        print(pc.iloc[-1].to_dict(), flush=True)
pc = pc.set_index("category").loc[CATEGORIES]
pc[["image_AUROC", "pixel_AUROC", "fit_s", "test_s_per_img", "n_train", "n_test"]]

#### PatchCore scores per image

For the intervals and the false-reject analysis below every image needs its own PatchCore score, and a threshold needs held-out good images. So PatchCore is fitted again on the same 80% of the good training images that the other methods use there, and the held-out fifth and the test images are scored. The scores are cached per category in `patchcore_scores/`; a first run takes about an hour on the CPU, mostly for choosing the coreset. Because of the smaller training set and the random coreset, these scores give an AUROC that differs by a few thousandths from the table above.

In [ ]:
from torch.utils.data import DataLoader
from anomalib.data import MVTecADDataset

PATCHCORE_SCORES = Path("patchcore_scores")


def patchcore_scores(ctg):
    """PatchCore scores of the held-out good images and of the test images, with PatchCore fitted on the same 80% of the
    good training images as the other methods in the operating-point analysis. Cached per category, a fit takes several minutes."""
    out = PATCHCORE_SCORES / f"{ctg}.npz"
    if out.exists():
        d = np.load(out)
        return d["s_val"], d["s_test"], d["y_test"]
    tr = sorted(p for p in PATH_ROOT.rglob("*.png")
                if p.parent.parent.parent.name == ctg and p.parent.parent.name == "train" and p.parent.name == "good")
    perm = np.random.default_rng(0).permutation(len(tr))
    n_val = len(tr) // 5
    val_paths, fit_paths = {tr[i] for i in perm[:n_val]}, {tr[i] for i in perm[n_val:]}  # same split as for the border margin

    def make_loader(split, keep=None):
        ds = MVTecADDataset(root=PATH_ROOT, category=ctg, split=split)
        if keep is not None:
            ds.samples = ds.samples[ds.samples.image_path.map(lambda s: Path(s) in keep)].reset_index(drop=True)
        return DataLoader(ds, batch_size=16, num_workers=0, collate_fn=ds.collate_fn)

    def collect(batches):
        out = [(b.pred_score.detach().cpu().numpy().ravel(), b.gt_label.detach().cpu().numpy().ravel())
               for item in batches for b in (item if isinstance(item, list) else [item])]
        return np.concatenate([o[0] for o in out]), np.concatenate([o[1] for o in out])

    model = Patchcore(post_processor=False, evaluator=False, visualizer=False)
    engine = Engine(accelerator="cpu", default_root_dir="anomalib_runs", logger=False, enable_progress_bar=False)
    t0 = time.perf_counter()
    engine.fit(model=model, train_dataloaders=make_loader("train", fit_paths))
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    s_val, _ = collect(engine.predict(model=model, dataloaders=make_loader("train", val_paths)))
    s_test, y_test = collect(engine.predict(model=model, dataloaders=make_loader("test")))
    s_per_img = (time.perf_counter() - t0) / (len(s_val) + len(s_test))  # includes loading the images
    PATCHCORE_SCORES.mkdir(exist_ok=True)
    np.savez(out, s_val=s_val, s_test=s_test, y_test=y_test, fit_s=fit_s, s_per_img=s_per_img)
    return s_val, s_test, y_test

### Comparing the methods

All methods side by side: a failure case, one summary table, how certain the numbers are, what the methods cost, how much training data they need, and what happens at a fixed false-reject rate.

#### Failure case: pill

`pill` is the category that no backbone or setting improves much. The same failure view as before, now for the baseline on `pill`: the good images with the highest scores in the top row, the defective images with the lowest scores in the bottom row. The last line prints the AUROC per defect type.

What the picture shows: the hottest spots in the *good* images are the red speckles of the pill, which are normal and differ from pill to pill, so a good pill always has some patches that no training patch matches. The weakest defects (`crack`, `scratch`, `faulty_imprint`) are subtle and score below that speckle noise, so no threshold separates them. Defects that change the whole appearance (`color`, `pill_type`, `combined`) are found much more reliably. This is the same conclusion as for the classical methods: the speckles are the difficulty, and a better description of the image does not remove them.

In [ ]:
memory_pill = build_knn_memory("pill")
show_failures("pill", lambda img: knn_map(memory_pill, img).max(), lambda img: knn_map(memory_pill, img))
print({k: round(v, 2) for k, v in deep["pill"]["per_defect"].items()})

#### Summary: all methods per category

Image AUROC on the test images of each category. Columns: the golden-template method with its final settings, the best of all classical variants tried above, the plain nearest-neighbor baseline, the same with the border ignored, the two ConvNeXt stages (with the border margin) and PatchCore. The border and ConvNeXt columns use margins chosen from good training images only.

Read it with care: the *best classical variant* was picked per category from several variants on the test images, which is a little optimistic for the classical side, and with so few test images the differences of 0.01 to 0.02 are within noise. The line above the table says which variant won where.

What it shows: PatchCore is clearly ahead only on `transistor` and `pill`, the two categories where the nearest-neighbor baselines were weakest. On `metal_nut` and `leather` it is equal or about one hundredth better. On `carpet` and `grid` the baseline with the border margin is just as good or slightly better (1.000 and 0.996 against 0.986 and 0.987), and with test sets this small those differences are noise.

The price is the time: fitting takes between 400 and 720 seconds per category against a few seconds for the random memory (see the speed table below). Which of the three things PatchCore adds (two layers, neighbourhood averaging, coreset) helps most is not tested here; that would need an ablation.

In [ ]:
last_step = list(STEPS)[-1]

best_cl = {}  # best classical variant per category: picked on the test images, so a little optimistic
for ctg_s in CATEGORIES:
    cand = {f"template: {st}": RESULTS[(st, ctg_s)]["auroc"] for st in STEPS if (st, ctg_s) in RESULTS}
    cand.update({m: r["auroc"] for m, r in tex.get(ctg_s, {}).items()})
    if ctg_s == "pill":
        cand.update({f"pill: {m}": r["auroc"] for m, r in pill.items()})
    best_cl[ctg_s] = max(cand.items(), key=lambda kv: kv[1])

cn = convnext_df.pivot(index="category", columns="features", values="AUROC ignoring border")
summary = pd.DataFrame({
    f"template ({last_step})": {c: round(RESULTS[(last_step, c)]["auroc"], 3) for c in CATEGORIES},
    "best classical variant": {c: round(v[1], 3) for c, v in best_cl.items()},
    "WRN50 kNN": {c: round(deep[c]["auroc"], 3) for c in CATEGORIES},
    "WRN50 + border": border_df.set_index("category")["AUROC ignoring border"],
    **{f"ConvNeXt {k.split(' (')[0]}": cn[k] for k in cn.columns},
    "PatchCore": pc["image_AUROC"].round(3),
}).loc[CATEGORIES]
print("which classical variant was best:", {c: v[0] for c, v in best_cl.items()})
summary

#### How sure are these numbers? Bootstrap intervals

The test sets are small (often 20 to 60 good and a few dozen defective images), so an AUROC is itself uncertain. For each method, the good and the defective test images are resampled with replacement 1000 times and the AUROC is recomputed; the table shows the AUROC with the range that contains 95% of the resampled values. Two methods differ for sure only if their intervals are clearly apart. PatchCore is the fit on 80% of the good images from the cell above, so its AUROC differs a little from the table. The intervals describe the choice of test images only, not the random memory subset or the choice of margin.

What it shows: against the golden template the deep methods are far ahead on `metal_nut`, `leather`, `carpet` and `grid`, with intervals that do not touch. On `transistor` and `pill` the intervals of the simple nearest-neighbor variants still overlap the template's, so there it is not proven that they are better; PatchCore's interval is clearly above the template's in every category. Between the deep variants most differences are not certain: the plain and the border-corrected baseline are clearly apart on `grid` and `carpet` but not on the objects, PatchCore's lead over the baseline is clear on `transistor` (0.98 to 1.00 against 0.80 to 0.94) and only just separates on `pill`.

In [ ]:
def bootstrap_auroc(y, s, n=1000, seed=0):
    """95% interval of the AUROC: resample the good and the defective images separately, with replacement."""
    rng = np.random.default_rng(seed)
    good, defect = s[y == 0], s[y == 1]
    labels_b = np.r_[np.zeros(len(good)), np.ones(len(defect))]
    out = [roc_auc_score(labels_b, np.r_[rng.choice(good, len(good)), rng.choice(defect, len(defect))]) for _ in range(n)]
    return np.percentile(out, [2.5, 97.5])


margin_w = border_df.set_index("category")["margin"]
margin_c = convnext_df.pivot(index="category", columns="features", values="margin")

rows_u = []
for ctg_u in CATEGORIES:
    idx = select(ctg_u, "test")
    y = (labels[idx] != "good").astype(int)
    memory = build_knn_memory(ctg_u)
    maps_w = np.array([knn_map(memory, X[i]) for i in idx])
    scores = {f"template ({last_step})": (y, RESULTS[(last_step, ctg_u)]["scores"]),
              "WRN50 kNN": (y, border_scores(maps_w, 0)),
              "WRN50 + border": (y, border_scores(maps_w, margin_w[ctg_u]))}
    for stage, name in [(3, "stage 2 (28x28, 192 ch)"), (5, "stage 3 (14x14, 384 ch)")]:
        mem_c, ch = memory_of(select(ctg_u, "train", good=True), stage)
        scores[f"ConvNeXt {name.split(' (')[0]} + border"] = (y, border_scores(knn_maps(idx, mem_c, ch, stage), margin_c.loc[ctg_u, name]))
    _, s_test_p, y_p = patchcore_scores(ctg_u)  # fitted on 80% of the good images, so a little different from the table above
    scores["PatchCore (80% fit)"] = (y_p.astype(int), s_test_p)
    for method, (y_m, s) in scores.items():
        lo, hi = bootstrap_auroc(y_m, s)
        rows_u.append({"category": ctg_u, "method": method, "text": f"{roc_auc_score(y_m, s):.3f} [{lo:.2f}, {hi:.2f}]"})

print("AUROC [95% bootstrap interval]; the number of test images is small, so the intervals are wide")
pd.DataFrame(rows_u).pivot(index="category", columns="method", values="text").loc[CATEGORIES]

#### Speed and cost

The same measurement for every method: one category (`grid`), the same 30 test images, one image at a time on the CPU, after a short warm-up. *Build* is the one-off cost of fitting the method on the good training images, *score* the time to judge one new image. The numbers depend on the machine and are only meant for comparing the methods with each other. In the run documented here, the classical methods need a few to a few tens of milliseconds per image, the deep nearest-neighbor methods tens to over a hundred; the memory search over tens of thousands of patches is the main cost. The last row is PatchCore through anomalib, read from its cached run. Its time per image includes loading the images and was measured in batches, so it is an upper limit compared with the other rows, and fitting takes minutes instead of seconds. Check the table after rerunning on your machine.

In [ ]:
ctg_t = "grid"  # one category; the cost per image hardly depends on the category
imgs_t = [X[i] for i in select(ctg_t, "test")[:30]]
print(f"{torch.get_num_threads()} CPU threads, GPU: {torch.cuda.is_available()}")


def time_method(build, make_score):
    """Build time (one-off, from the good training images) and scoring time per image, same images for every method."""
    t0 = time.perf_counter()
    obj = build()
    build_s = time.perf_counter() - t0
    fn = make_score(obj)
    for im in imgs_t[:3]:
        fn(im)  # warm-up
    t0 = time.perf_counter()
    for im in imgs_t:
        fn(im)
    return build_s, (time.perf_counter() - t0) / len(imgs_t)


def convnext_score(obj, stage):
    memory, ch = obj
    return lambda img: torch.cdist(torch.from_numpy(convnext_features([img], stage)[0].reshape(-1, ch)), memory).min(dim=1).values.max().item()


tr_t = select(ctg_t, "train", good=True)
methods = {
    "template z + ECC (256 px)": (lambda: build_model(ctg_t, res=256, align=True), lambda m: (lambda img: score(m, img))),
    "Fourier notch": (lambda: None, lambda _: score_fourier),
    "Gabor + Mahalanobis": (lambda: make_gabor_score(ctg_t), lambda f: f),
    "WRN50 kNN (layer 2)": (lambda: build_knn_memory(ctg_t), lambda mem: (lambda img: knn_map(mem, img).max())),
    "ConvNeXt kNN (stage 2)": (lambda: memory_of(tr_t, 3), lambda o: convnext_score(o, 3)),
    "ConvNeXt kNN (stage 3)": (lambda: memory_of(tr_t, 5), lambda o: convnext_score(o, 5)),
}
timing = pd.DataFrame(
    {n: dict(zip(["build (s)", "score (ms/img)"], time_method(*fns))) for n, fns in methods.items()}
).T
timing["score (ms/img)"] *= 1000
timing = timing.round(1)
timing["images/s"] = (1000 / timing["score (ms/img)"]).round(0)
if (PATCHCORE_SCORES / f"{ctg_t}.npz").exists():  # PatchCore: times of its cached run, the prediction time includes loading the images
    d_pc = np.load(PATCHCORE_SCORES / f"{ctg_t}.npz")
    timing.loc["PatchCore (anomalib, incl. image loading)"] = [round(float(d_pc["fit_s"]), 1), round(float(d_pc["s_per_img"]) * 1000, 1), round(1 / float(d_pc["s_per_img"]), 0)]
timing

#### How many good images does it need?

On a real line there are often only a handful of golden samples. Here the training set is reduced to 5, 10, 25 and 50 randomly chosen good images (three different random subsets each, the bars show the spread), and the nearest-neighbor baseline with the border margin is compared with the golden-template method with ECC alignment. The border margin is the one chosen above from the full training set. The last point of each curve uses all good images. This takes several minutes, mostly for the template alignment.

What to look for: the textures (`leather`, `carpet`) are already perfect with 5 images for the nearest-neighbor method, because a texture has no layout to learn. The parts (`metal_nut`, `grid`, `transistor`) need more, and gain most between 5 and 25 images. The template method does not improve with more images (on `grid` it even gets worse), so its limit is the method, not the data. Only for `transistor` is it better than the deep method with very few images (about 10 or fewer). The random memory subset adds noise of its own, which is clearly visible on `pill`; this is what PatchCore's coreset sampling is meant to remove.

In [ ]:
def build_template(idxs, res=256):
    """Golden template (per-pixel mean and std after ECC alignment) from the given good training images."""
    train = [prep(X[i], res) for i in idxs]
    tmpl = train[0]
    train = np.stack([align_ecc(g, tmpl)[0] for g in train])
    return dict(res=res, align=True, aligner=align_ecc, mean=train.mean(0), std=train.std(0) + STD_FLOOR, tmpl=tmpl, mask=None)


sizes, reps = [5, 10, 25, 50, None], 3  # None = all good training images; random subsets, repeated `reps` times
curve = []
for ctg_n in CATEGORIES:
    tr = select(ctg_n, "train", good=True)
    idx = select(ctg_n, "test")
    y = (labels[idx] != "good").astype(int)
    f_tr = cnn_features(list(X[tr]))                                          # features of every good training image
    f_te = torch.from_numpy(cnn_features([X[i] for i in idx]).reshape(len(idx), -1, 512))
    m = int(margin_w[ctg_n])
    for n in sizes:
        k = len(tr) if n is None else min(n, len(tr))
        for rep in range(1 if k == len(tr) else reps):
            sub = np.sort(np.random.default_rng(rep).choice(len(tr), size=k, replace=False))  # sorted: with all images, same memory as the baseline
            f = f_tr[sub].reshape(-1, 512)
            mem = torch.from_numpy(f[np.random.default_rng(0).choice(len(f), size=min(40000, len(f)), replace=False)])
            maps = np.array([torch.cdist(q, mem).min(dim=1).values.reshape(28, 28).numpy() for q in f_te])
            model = build_template(tr[sub])
            curve.append({"category": ctg_n, "n": k, "rep": rep,
                          "WRN50 kNN + border": roc_auc_score(y, border_scores(maps, m)),
                          "template z + ECC": roc_auc_score(y, [score(model, X[i]) for i in idx])})
    print(ctg_n, "done", flush=True)

curve = pd.DataFrame(curve)
fig, ax = plt.subplots(2, 3, figsize=(13, 6.5), sharey=True)
for a, ctg_n in zip(ax.ravel(), CATEGORIES):
    d = curve[curve.category == ctg_n].groupby("n")
    for method in ["WRN50 kNN + border", "template z + ECC"]:
        a.errorbar(d[method].mean().index, d[method].mean(), yerr=d[method].std().fillna(0), marker="o", capsize=3, label=method)
    a.set_xscale("log"); a.set_title(ctg_n); a.set_xlabel("good training images")
ax[0, 0].set_ylabel("image AUROC"); ax[1, 0].set_ylabel("image AUROC"); ax[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()
curve.groupby(["category", "n"]).mean(numeric_only=True).drop(columns="rep").round(3)

#### What the AUROC hides: defects caught at a fixed false-reject rate

The AUROC compares all possible thresholds. A line needs one threshold, set before it has seen the parts. Here the threshold is a quantile of the scores of the held-out good training images (the same fifth that was used for the border margin, 42 to 56 images per category), aimed at 5% and at 1% false rejects. The first two tables show the share of defective test images above the threshold and, in brackets, the false-reject rate this threshold really gives on the good test images. The third shows, for comparison, how many defects each method would catch at 5% false rejects with the best possible threshold; it uses the test labels, so it is a ceiling and not a procedure. PatchCore is the fit on 80% of the good images (cached, see above). The cell needs several minutes (about 15 in the test run, with PatchCore cached).

What to look at:

- A high AUROC does not mean most defects are caught at a usable threshold. On `transistor` and `pill` the nearest-neighbor variants catch roughly half of the defects or fewer at 5% false rejects, PatchCore catches 100% and 74%. On `metal_nut` and the textures most methods catch 90% or more.
- The false-reject rate rarely hits its target. With 40 to 60 good images the threshold is itself a noisy estimate, and the good test images are only 20 to 60, so each one is worth several percent. PatchCore overshoots most (14 to 36% instead of 5% on five of six categories): the held-out training images score lower than the good test images, so its threshold sits too low, and part of its high catch rate is paid for with false rejects. The golden template on `leather` even rejects every good test image.
- Because of that, the ceiling table is the fairer comparison of the ranking itself: PatchCore leads clearly only on `transistor` (95% against 28 to 62%), on `pill` it is 78% against 74% for the baseline, and elsewhere the methods are about equal. Setting the threshold is a separate problem from ranking the images well.

In [ ]:
def operating_point(scores_val, scores_good, scores_defect, target_frr):
    """Threshold from held-out good images only. Returns the share of defects caught and the false-reject rate on the good test images."""
    thr = np.quantile(scores_val, 1 - target_frr, method="higher")
    return (scores_defect > thr).mean(), (scores_good > thr).mean()


from sklearn.metrics import roc_curve

targets = (0.05, 0.01)  # false-reject rates the threshold is aimed at
res_op, res_ceiling = [], []
for ctg_o in CATEGORIES:
    tr = select(ctg_o, "train", good=True)
    perm = np.random.default_rng(0).permutation(len(tr))
    n_val = len(tr) // 5
    val_i, fit_i = tr[perm[:n_val]], tr[perm[n_val:]]   # the same split as for the border margin
    idx = select(ctg_o, "test")
    is_def = labels[idx] != "good"
    split_test = lambda s_test: (s_test[~is_def], s_test[is_def])
    sc = {}  # method -> (scores of the held-out good images, of the good test images, of the defective test images)

    f = cnn_features(list(X[fit_i])).reshape(-1, 512)
    mem = torch.from_numpy(f[np.random.default_rng(0).choice(len(f), size=min(40000, len(f)), replace=False)])
    mv = np.array([knn_map(mem, X[i]) for i in val_i])
    mt = np.array([knn_map(mem, X[i]) for i in idx])
    sc["WRN50 kNN"] = (border_scores(mv, 0), *split_test(border_scores(mt, 0)))
    sc["WRN50 + border"] = (border_scores(mv, margin_w[ctg_o]), *split_test(border_scores(mt, margin_w[ctg_o])))
    for stage, name in [(3, "stage 2 (28x28, 192 ch)"), (5, "stage 3 (14x14, 384 ch)")]:
        mem_c, ch = memory_of(fit_i, stage)
        m = margin_c.loc[ctg_o, name]
        sc[f"ConvNeXt {name.split(' (')[0]} + border"] = (border_scores(knn_maps(val_i, mem_c, ch, stage), m),
                                                          *split_test(border_scores(knn_maps(idx, mem_c, ch, stage), m)))
    model = build_template(fit_i, res=512)
    sc["template z + ECC (512 px)"] = (np.array([score(model, X[i]) for i in val_i]), *split_test(np.array([score(model, X[i]) for i in idx])))
    s_val_p, s_test_p, y_p = patchcore_scores(ctg_o)  # fitted on the same fit_i; cached
    sc["PatchCore"] = (s_val_p, s_test_p[y_p == 0], s_test_p[y_p == 1])

    for method, (s_val, s_good, s_def) in sc.items():
        for t in targets:
            caught, frr = operating_point(np.asarray(s_val), s_good, s_def, t)
            res_op.append({"category": ctg_o, "method": method, "target": t, "caught": caught, "false rejects": frr})
        fpr, tpr, _ = roc_curve(np.r_[np.zeros(len(s_good)), np.ones(len(s_def))], np.r_[s_good, s_def])
        res_ceiling.append({"category": ctg_o, "method": method, "ceiling": tpr[fpr <= 0.05].max()})  # best possible threshold, uses the test labels
    print(ctg_o, "done", flush=True)

res_op = pd.DataFrame(res_op)
for t in targets:
    d = res_op[res_op.target == t]
    d = d.assign(text=d.apply(lambda r: f"{r['caught']:.0%} ({r['false rejects']:.0%})", axis=1))
    print(f"Defects caught (false-reject rate on the good test images in brackets), threshold aimed at {t:.0%} false rejects")
    display(d.pivot(index="category", columns="method", values="text").loc[CATEGORIES])

res_ceiling = pd.DataFrame(res_ceiling)
print("Defects caught at 5% false rejects with the best possible threshold (uses the test labels: a ceiling, not a procedure)")
display(res_ceiling.assign(text=res_ceiling["ceiling"].map("{:.0%}".format)).pivot(index="category", columns="method", values="text").loc[CATEGORIES])

# Write-up: Can a pretrained network beat a golden template? Anomaly detection on MVTec AD

## Summary

I spent 18 years around inspection equipment, where "find the part that does not look like the good ones" is solved with golden templates, filters and a lot of tuning. I wanted to see how far a pretrained neural network gets on the same task with no defect examples at all, only good images. On six MVTec AD categories, a nearest-neighbor method on pretrained features beat my golden-template pipeline clearly on four of them (a tuned classical method came close on two of those), and PatchCore was the best overall, but it costs about a hundred times more to fit. What surprised me: the biggest single gain (grid, AUROC 0.77 to 1.00) came from understanding one artifact at the image border, not from a better model. The fix has a price: a defect lying only in the ignored border cannot be found (one of the 57 defective grid images). Built with Claude as a coding assistant; the notebook is the full record.

## Question

Can unsupervised deep anomaly detection beat a classical golden-template method, and what does it cost in speed and in false rejects?

## Data

[MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad), six categories: transistor, pill, metal_nut (objects) and leather, carpet, grid (textures). Training uses good images only; the test sets hold good and defective images (78 to 167 per category). The dataset is licensed CC BY-NC-SA 4.0 (attribution, non-commercial use, share-alike) and must be cited (Bergmann et al., CVPR 2019). The figures in this write-up show MVTec AD images, so the same licence applies to them.

To get the data, download MVTec AD from the page above (it is a manual download, the licence is accepted there) and keep one folder per category in the layout `<category>/train/good`, `<category>/test/<good or defect type>` and `<category>/ground_truth/...`. Only the six categories above are used, the notebook ignores any others. Point the environment variable `MVTEC_AD_ROOT` (or `PATH_ROOT` in the first code cell) at the folder that holds the category folders. The automatic download in `anomalib` did not work when I tried it.

## Methods

1. **Classical baseline.** Aligned golden template (per-pixel mean and standard deviation of the good images, ECC alignment, blurred z-map, score = a high percentile). For textures: high-pass, Fourier notch (remove the repeating pattern), Gabor energy with Mahalanobis distance. For the pill, color in Lab space.
2. **Simple deep baseline.** Features of a pretrained Wide ResNet50 (layer 2, a 28 x 28 grid of 512-number patch descriptions), a memory of good patches, score = distance of the worst test patch to its nearest good patch. Then the same with ConvNeXt-tiny.
3. **PatchCore** through anomalib, with its defaults (two layers, neighborhood averaging, 10% coreset).

All numbers are image AUROC on the test images, on the CPU, no GPU. AUROC is the usual metric for this benchmark, needs no threshold and does not depend on how many defective images the test set has, so the results can be compared with published ones. It is not what a line needs, which is one threshold and a price on every false reject and every escape; the false-reject analysis below looks at that.

## Results

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead><tr>
<th style="text-align: left;">Category</th>
<th style="text-align: left;">Golden template</th>
<th style="text-align: left;">Best classical variant</th>
<th style="text-align: left;">NN baseline</th>
<th style="text-align: left;">+ ignore border</th>
<th style="text-align: left;">ConvNeXt (best stage)</th>
<th style="text-align: left;">PatchCore</th>
</tr></thead>
<tbody>
<tr><td style="text-align: left;">transistor</td><td style="text-align: left;">0.83</td><td style="text-align: left;">0.84</td><td style="text-align: left;">0.88</td><td style="text-align: left;">0.88</td><td style="text-align: left;">0.95</td><td style="text-align: left;"><b>0.995</b></td></tr>
<tr><td style="text-align: left;">pill</td><td style="text-align: left;">0.75</td><td style="text-align: left;">0.80</td><td style="text-align: left;">0.85</td><td style="text-align: left;">0.85</td><td style="text-align: left;">0.88</td><td style="text-align: left;"><b>0.95</b></td></tr>
<tr><td style="text-align: left;">metal_nut</td><td style="text-align: left;">0.66</td><td style="text-align: left;">0.66</td><td style="text-align: left;">0.96</td><td style="text-align: left;">0.96</td><td style="text-align: left;">0.99</td><td style="text-align: left;"><b>1.00</b></td></tr>
<tr><td style="text-align: left;">leather</td><td style="text-align: left;">0.55</td><td style="text-align: left;">0.99</td><td style="text-align: left;">1.00</td><td style="text-align: left;">1.00</td><td style="text-align: left;">1.00</td><td style="text-align: left;">1.00</td></tr>
<tr><td style="text-align: left;">carpet</td><td style="text-align: left;">0.29</td><td style="text-align: left;">0.75</td><td style="text-align: left;">0.98</td><td style="text-align: left;">1.00</td><td style="text-align: left;">1.00</td><td style="text-align: left;">0.99</td></tr>
<tr><td style="text-align: left;">grid</td><td style="text-align: left;">0.55</td><td style="text-align: left;">0.95</td><td style="text-align: left;">0.77</td><td style="text-align: left;">1.00</td><td style="text-align: left;">0.99</td><td style="text-align: left;">0.99</td></tr>
</tbody>
</table>

*"Best classical variant" and the best ConvNeXt stage were picked per category on the test images, so they are slightly optimistic.*

- **The deep methods win clearly on metal_nut, leather, carpet and grid.** With 95% bootstrap intervals the template's range does not touch theirs. On transistor and pill the intervals of the simple nearest-neighbor methods overlap the template's, so "deep is better" is not proven for them there; PatchCore's interval is above the template's in every category.
- **The template method fails on textures.** On carpet its AUROC is below 0.5, worse than a coin flip. A texture has no fixed layout, so good images differ from the template more than defects do.
- **PatchCore is clearly ahead only on transistor and pill**, where the simpler methods were weakest. Elsewhere it is equal or within noise.

Examples for `transistor` with the nearest-neighbor baseline: a good image, a defect that is found, one that is missed and a false alarm. The threshold comes from held-out good images, aimed at 5% false rejects; the figure is also saved as `figure_examples.png`.

In [ ]:
ctg_f = "transistor"
tr = select(ctg_f, "train", good=True)
perm = np.random.default_rng(0).permutation(len(tr))
n_val = len(tr) // 5
val_i, fit_i = tr[perm[:n_val]], tr[perm[n_val:]]  # same split as for the border margin and the thresholds
f = cnn_features(list(X[fit_i])).reshape(-1, 512)
memory_f = torch.from_numpy(f[np.random.default_rng(0).choice(len(f), size=min(40000, len(f)), replace=False)])
thr = np.quantile([knn_map(memory_f, X[i]).max() for i in val_i], 0.95, method="higher")  # aimed at 5% false rejects

idx = select(ctg_f, "test")
maps_f = {i: knn_map(memory_f, X[i]) for i in idx}
s_f = np.array([maps_f[i].max() for i in idx])
is_def = labels[idx] != "good"
good_j, def_j = np.where(~is_def)[0], np.where(is_def)[0]
picks = {
    "good image": idx[good_j[np.argsort(s_f[good_j])[len(good_j) // 2]]],  # the median good image
    "defect found": idx[def_j[np.argmax(s_f[def_j])]],
    "defect missed": idx[def_j[np.argmin(s_f[def_j])]],
    "false alarm": idx[good_j[np.argmax(s_f[good_j])]],  # the highest-scoring good image
}
vmax = max(maps_f[i].max() for i in picks.values())

fig, ax = plt.subplots(2, 4, figsize=(15, 7.5))
for c, (name, i) in enumerate(picks.items()):
    up = cv2.resize(maps_f[i], X[i].shape[1::-1], interpolation=cv2.INTER_LINEAR)
    ax[0, c].imshow(X[i])
    ax[1, c].imshow(X[i]); ax[1, c].imshow(up, cmap="jet", alpha=0.5, vmin=0, vmax=vmax)
    verdict = "rejected" if maps_f[i].max() > thr else "passed"
    ax[0, c].set_title(f"{name}\n{labels[i]}, score {maps_f[i].max():.2f} ({verdict})", fontsize=10)
for a in ax.ravel():
    a.axis("off")
fig.suptitle(f"{ctg_f}, nearest-neighbor baseline, threshold {thr:.2f} from held-out good images (aimed at 5% false rejects)")
plt.tight_layout()
plt.savefig("figure_examples.png", dpi=110)
plt.show()

## Failure cases

- **Grid, the border.** The first nearest-neighbor result on grid was poor (0.77). The failure view showed that the highest scores of *good* images sat on the edges and corners; the network's padding makes edge features unlike anything in the memory. Ignoring a two-patch border took the AUROC to 0.996. The margin was chosen from held-out good training images only, with no test image involved, and it came out as 0 for the object categories, where nothing needed fixing. A cap on the margin is needed on coarse feature grids (ConvNeXt stage 3), where a larger margin cut away too much of the image. The price of the margin is a blind spot at the edge: of the 57 defective grid images, one has its whole defect inside the ignored ring and cannot be found (see the edge check above).
- **Pill.** No backbone or setting improves it much. The brightest spots in good pills are their normal red speckles, and subtle defects (crack, scratch, faulty imprint, the worst at 0.64) score below that noise. Defects that change the whole look (color, wrong pill type) are found reliably.
- **Transistor.** In the figure above, the most clearly missed defect (labelled `misplaced`) is a board with no transistor on it at all. Every patch of an empty board looks like normal board, so a method that compares local patches has nothing to flag; a missing part is a different kind of defect from a damaged surface. On a line a presence check like this is usually a quick conventional step of its own, so the patch-based method does not have to cover it. The highest-scoring good image is hot at the lead tips and at one image corner, which is normal variation.
- **Which features.** The best scale depends on the defect: coarse features help the parts, fine features are needed for grid, where defects are small.

## What this means on a real line

- **Cycle time.** On a line the hardware is sized, usually with enough GPU power, so that image acquisition sets the inspection speed and not the processing. Processing can still become the bottleneck, depending on the method and on how many potential defects have to be evaluated: anywhere from milliseconds to a few hundred milliseconds per image. In IC inspection, for example, one field of view holds several devices, each of them aligned and inspected, and that time covers the whole image. *In this study* every image holds a single part (CPU only, no GPU tested): the classical methods need 5 to 20 ms per image, nearest neighbor 60 to 140 ms, PatchCore 0.3 to 0.8 s including image loading. With several parts per image at the same detail the time would grow roughly with their number (an estimate, not measured). Fitting PatchCore takes minutes per category; that is done once and offline, but it matters when the model has to be rebuilt after a change.
- **Golden samples and tuning.** What customers provide varies: sometimes golden (good) samples, sometimes reject samples, with the models tuned on what the system has to detect. The usual flow is to run on production lots and check for overkill (good parts rejected), because new variations always appear; to tune if needed; and then to re-check the known defect samples. *In this study* there were only good images, no reject samples and no tuning loop. Five good images were enough for the textures; the parts needed about 25 to 50.
- **False reject against escape.** The two mistakes do not cost the same. An escape costs a multiple of a false reject and is hard to price: batches that end customers send back, or lots that must be rescanned after the model was tuned to catch it. That argues for a threshold that leans towards catching defects; the price of each mistake decides how far. What counts is the rate per part, because each part is accepted or rejected on its own; since every image here holds one part, the rates in this study are per part. Only if a flagged part triggers action on the whole image (re-inspecting or scrapping a strip or tray, a person reviewing the image), the false rejects add up: with a per-part rate p, an image with n good parts is flagged with probability 1 − (1 − p)ⁿ, which is about a third of the images at p = 5% and n = 8. *In this study* thresholds came from held-out good images, aimed at 5% false rejects, and the two mistakes were not weighed. The best nearest-neighbor variants caught 95 to 100% of the defects on four categories but only 45 to 65% on transistor and pill, where PatchCore caught 100% and 74%. The false-reject rate often missed its target (0 to 36%): setting the threshold is a separate problem from ranking images well, and the overkill check on new lots is what catches it.
- **Yield target.** IC inspection often aims at 99% yield or more, so the false-reject budget is well below the 5% used first and the 1% table is the more relevant one. There the best nearest-neighbor variants catch about half of the defects on transistor and 28% on pill, PatchCore 92% and 64%. The thresholds also missed the 1% target on several categories (up to 29% on carpet for PatchCore). And a rate of 1% cannot be measured with 20 to 60 good test images: showing it with confidence takes about 300 good parts without a single reject (rule of three: the 95% upper bound is 3 divided by the number of parts).
- **Where a benchmark result would not transfer.** MVTec images are clean, evenly lit and well fixtured. On a real line much of that can be kept in check: setups are maintained, and drift in lighting, camera or fixture is caught with calibration and calibration checks. Part variation is the hardest: it is the least predictable and the hardest to check, and a sudden change in incoming material is very hard to handle. This benchmark tests none of it: one fixed set of parts, no new lots, no threshold drift, no recalibration.

## Reproduce

Python 3.14 on Windows, CPU only (8 threads), no GPU. The package versions are in `requirements.txt` (`pip install -r requirements.txt`; `anomalib` is also installed by a cell near the top if it is missing). Download MVTec AD as described in the Data section, point `MVTEC_AD_ROOT` (or `PATH_ROOT` in the first code cell) at it, then run the notebook top to bottom. Rough run times on this machine: loading the images about 2 minutes, the classical experiments a few minutes, the deep experiments tens of minutes, the bootstrap and training-size experiments about 20 minutes each, PatchCore about one hour for the table and about one more for the per-image scores (cached in `patchcore_results.csv` and `patchcore_scores/`, so a rerun reads them unless you delete them). Seeds are fixed except PatchCore's random coreset, which changes the result by a few thousandths between runs.

## Limitations and next steps

- One object per image: there is no field of view with several parts, no alignment of each part and no per-part threshold. The false-reject rates here are per part because an image holds one part; the time per image would grow with the number of parts.
- Six of the fifteen categories, small test sets (differences of 0.01 to 0.02 are noise), one seed, one machine.
- The best classical variant and the best ConvNeXt stage were chosen on the test images. The border margin was chosen without them, but I only looked for it after the first poor result on grid.
- PatchCore's interval and false-reject numbers come from a separate fit on 80% of the good images; no ablation of its three additions, ConvNeXt-tiny only.
- Checked since: defects at the image edge. Ignoring the border hides one defect image for sure (grid, the whole defect lies in the ring) and part of the defect in 21 others; none that was caught without the margin is lost (masks for all six categories, scores for grid only, with an optimistic threshold). A defect at the very edge is still a blind spot. Next, with this data: a PatchCore ablation, and a test of how the methods cope with a shifted or rotated part.
- Next, on a larger dataset with many more good parts: (1) several parts per image, or a hybrid where the part is found first and the inspection works on the cropped part; (2) the effect of batch variations, simulated by augmenting the images (lighting, colour, surface) to mimic changes of the parts; (3) whether any of the methods can reach a high-yield target of 99% or more, which needs enough good parts to measure a false-reject rate of 1% or below; (4) other deep methods, judged at the false-reject target and not by AUROC alone: EfficientAD (built for low latency), PaDiM (a Gaussian for every image position, which fits aligned parts), methods on transformer features such as AnomalyDINO and Dinomaly (few-shot, strong accuracy) and DRAEM (trained with synthetic defects, related to the augmentation idea).
- Further ideas, not yet decided: (1) using a few reject samples to tune the threshold, as in the usual customer flow of checking overkill, tuning and re-checking the defect samples, and measuring how many defect samples it takes to help; (2) a threshold method that keeps its false-reject target on new lots, with more good parts for the threshold, recalibration per lot and monitoring for drift, judged by how close the real false-reject rate comes to the target.
